# PCB validation error diagnosis — no training

Use the same version-2 dataset and the full-board experiment's checkpoint ZIP. If its Kaggle working folder still exists, the default path below works. Otherwise attach the downloaded ZIP as a private dataset and set CHECKPOINT to its path. Keep the original checkpoint downloaded before ending the session.

This notebook compares the old baseline and selected candidate on the 120 source validation boards. It measures missed defects and unmatched predictions at a fixed confidence of 0.25, with IoU 0.5 matching. It also records class/box details for diagnosing errors. No training or development-test evaluation occurs. These diagnostic counts are not mAP and use a different operating point from the training report.


In [ ]:
import sys, subprocess, importlib.metadata as meta
from pathlib import Path
from packaging.requirements import Requirement
# Preserve Kaggle's CUDA stack and NumPy ABI while resolving other dependencies.
constraints = Path('/kaggle/working/pcb_runtime_constraints.txt')
constraints.write_text(''.join(f'{name}=={meta.version(name)}\n' for name in ['torch', 'torchvision', 'numpy']))
def satisfied():
    try:
        if meta.version('ultralytics') != '8.4.142': return False
        for text in meta.requires('ultralytics'):
            req = Requirement(text)
            if req.marker and not req.marker.evaluate({'extra': ''}): continue
            if meta.version(req.name) not in req.specifier: return False
        return True
    except meta.PackageNotFoundError:
        return False
if not satisfied():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.142', '--constraint', str(constraints)], check=True)
assert satisfied(), 'Ultralytics dependencies are incomplete'
import torch, torchvision


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, zipfile, os
WORK = Path('/kaggle/working/pcb_phase2')
if not WORK.exists():
    INPUT = Path('/kaggle/input')
    records = list(INPUT.rglob('package_hashes.json'))
    archives = list(INPUT.rglob('pcb_revamp_phase2.zip'))
    if len(records) == 1:
        shutil.copytree(records[0].parent, WORK)
    elif not records and len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as z:
            names = z.namelist()
            if len(names) != len(set(names)): raise ValueError('Duplicate archive entries')
            for info in z.infolist():
                p = PurePosixPath(info.filename)
                if p.is_absolute() or '..' in p.parts or '\\' in info.filename or ((info.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('Unsafe archive path')
            if z.testzip() is not None: raise ValueError('Damaged archive')
            z.extractall(WORK)
    else:
        raise RuntimeError('Attach exactly one Phase 2 package (expanded dataset or ZIP).')

def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
EXPECTED_MANIFEST = '8687e5c1a3b91d3953abe7b10ebe8c793adfdb46f7280be9b8545f9f7ceb1419'
assert file_hash(WORK/'package_hashes.json') == EXPECTED_MANIFEST, 'Wrong package version'
record = json.loads((WORK/'package_hashes.json').read_text())
for name, expected in record['sha256'].items():
    path = (WORK/name).resolve()
    assert path.is_relative_to(WORK.resolve()), 'Unsafe package path'
    assert file_hash(path) == expected, 'Changed asset: '+name
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/pcb_yolo_settings'
os.environ['WANDB_MODE'] = 'disabled'
EMBEDDED_TOOLS = {'__init__.py': {'base64': 'IiIiUGhhc2VkLCBub24tZGVzdHJ1Y3RpdmUgUENCIG1vZGVsIHJldmFtcCB0b29saW5nLiIiIgo=', 'sha256': '35648be55b79bf5b9573ee55b6bd27e7e3c3942a17836c70fc99e29a3f863576'}, 'runtime.py': {'base64': 'IiIiUG9ydGFibGUsIG9mZmxpbmUgcGFja2FnZSB2ZXJpZmljYXRpb24gYW5kIHdyaXRhYmxlIGRhdGFzZXQgY29uZmlndXJhdGlvbi4iIiIKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmltcG9ydCB6aXBmaWxlCgpOQU1FUyA9IFsibWlzc2luZ19ob2xlIiwgIm1vdXNlX2JpdGUiLCAib3Blbl9jaXJjdWl0IiwgInNob3J0IiwgInNwdXIiLCAic3B1cmlvdXNfY29wcGVyIl0KCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDEwMjQqMTAyNCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9wYXRoKHJvb3QsIHJlbGF0aXZlKToKICAgIHBhdGggPSBQdXJlUG9zaXhQYXRoKHJlbGF0aXZlKQogICAgaWYgbm90IHJlbGF0aXZlIG9yIHBhdGguaXNfYWJzb2x1dGUoKSBvciAiLi4iIGluIHBhdGgucGFydHMgb3IgIlxcIiBpbiByZWxhdGl2ZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zYWZlIHBhdGg6IHtyZWxhdGl2ZX0iKQogICAgcmVzdWx0ID0gKFBhdGgocm9vdCkvcmVsYXRpdmUpLnJlc29sdmUoKQogICAgaWYgbm90IHJlc3VsdC5pc19yZWxhdGl2ZV90byhQYXRoKHJvb3QpLnJlc29sdmUoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBhdGggZXNjYXBlcyByb290OiB7cmVsYXRpdmV9IikKICAgIHJldHVybiByZXN1bHQKCgpkZWYgdmVyaWZ5X3ZpZXdzKHJvb3QsIG1hbmlmZXN0KToKICAgICIiIlZhbGlkYXRlIGZpbGVzLCBkZWNvZGVkIGdlb21ldHJ5LCBwYXJlbnQgc3BsaXRzLCBncm91cCBzcGxpdHMgYW5kIHRheG9ub215LiIiIgogICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICBpZiBtYW5pZmVzdFsibmFtZXMiXSAhPSBOQU1FUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDbGFzcyBvcmRlciBjaGFuZ2VkLiIpCiAgICBzZWVuLCBwYXJlbnRzLCBncm91cHMsIGhhc2hlcywgY2xhc3NlcyA9IHNldCgpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpCiAgICBmb3Igcm93IGluIG1hbmlmZXN0WyJ2aWV3cyJdOgogICAgICAgIGRvbWFpbiwgc3BsaXQsIGtpbmQgPSByb3dbImRhdGFzZXQiXSwgcm93WyJzcGxpdCJdLCByb3dbImtpbmQiXQogICAgICAgIGlmIGRvbWFpbiBub3QgaW4gKCJzb3VyY2UiLCAidGFyZ2V0Iikgb3Igc3BsaXQgbm90IGluICgidHJhaW4iLCAidmFsIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVua25vd24gZG9tYWluIG9yIGRldmVsb3BtZW50LXRlc3QgZGF0YSBpbiBwYWNrYWdlLiIpCiAgICAgICAgaWYga2luZCBub3QgaW4gKCJmdWxsIiwgImNyb3AiLCAicmV2aWV3ZWRfYmFja2dyb3VuZCIpIG9yIChraW5kICE9ICJmdWxsIiBhbmQgKGRvbWFpbiAhPSAic291cmNlIiBvciBzcGxpdCAhPSAidHJhaW4iKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkRlcml2ZWQgdmlld3MgbXVzdCBiZSBzb3VyY2UgdHJhaW5pbmcgaW1hZ2VzLiIpCiAgICAgICAgaWYgbm90IHJvd1sicGFyZW50X2lkIl0uc3RhcnRzd2l0aChkb21haW4rIi8iKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUGFyZW50IGJlbG9uZ3MgdG8gYW5vdGhlciBkb21haW4uIikKICAgICAgICBpZiBraW5kID09ICJyZXZpZXdlZF9iYWNrZ3JvdW5kIiBhbmQgbm90IHJvdy5nZXQoInJldmlldyIpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJCYWNrZ3JvdW5kIGhhcyBubyByZXZpZXcgcmVjb3JkLiIpCiAgICAgICAgZm9yIGtleSBpbiAoImltYWdlIiwgImxhYmVsIik6CiAgICAgICAgICAgIGlmIHJvd1trZXldIGluIHNlZW46CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUgZmlsZSByZWZlcmVuY2UuIikKICAgICAgICAgICAgc2Vlbi5hZGQocm93W2tleV0pCiAgICAgICAgICAgIHBhdGggPSBzYWZlX3BhdGgocm9vdCwgcm93W2tleV0pCiAgICAgICAgICAgIGlmIGRpZ2VzdChwYXRoKSAhPSByb3dba2V5KyJfc2hhMjU2Il06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQ2hhbmdlZCB7a2V5fToge3Jvd1trZXldfScpCiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKHNhZmVfcGF0aChyb290LCByb3dbImltYWdlIl0pKSBhcyBpbToKICAgICAgICAgICAgaW0ubG9hZCgpCiAgICAgICAgICAgIGlmIGltLnNpemUgIT0gKHJvd1sid2lkdGgiXSwgcm93WyJoZWlnaHQiXSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJXcm9uZyBpbWFnZSBkaW1lbnNpb25zLiIpCiAgICAgICAgcGFyc2VkID0gW1tmbG9hdCh2KSBmb3IgdiBpbiBsaW5lLnNwbGl0KCldIGZvciBsaW5lIGluIHNhZmVfcGF0aChyb290LHJvd1sibGFiZWwiXSkucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgICAgICBpZiBsZW4ocGFyc2VkKSAhPSBsZW4ocm93WyJsYWJlbHMiXSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1hbmlmZXN0L2xhYmVsIGNvdW50IG1pc21hdGNoLiIpCiAgICAgICAgaWYgKG5vdCBwYXJzZWQpICE9IChraW5kID09ICJyZXZpZXdlZF9iYWNrZ3JvdW5kIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgZW1wdHkgb3Igbm9uZW1wdHkgYW5ub3RhdGlvbnMuIikKICAgICAgICBmb3IgdmFsdWVzLCBleHBlY3RlZCBpbiB6aXAocGFyc2VkLCByb3dbImxhYmVscyJdKToKICAgICAgICAgICAgaWYgbGVuKHZhbHVlcykgIT0gNSBvciBub3QgYWxsKG1hdGguaXNmaW5pdGUodikgZm9yIHYgaW4gdmFsdWVzKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgbGFiZWwgcm93LiIpCiAgICAgICAgICAgIGMsIHgsIHksIHcsIGggPSB2YWx1ZXMKICAgICAgICAgICAgYWxsb3dlZCA9IHJhbmdlKDYpIGlmIGRvbWFpbiA9PSAic291cmNlIiBlbHNlIHJhbmdlKDEsNikKICAgICAgICAgICAgaWYgYyAhPSBpbnQoYykgb3IgaW50KGMpIG5vdCBpbiBhbGxvd2VkIG9yIG1pbih3LGgpIDw9IDAgb3IgbWluKHgtdy8yLHktaC8yKSA8IC0xZS02IG9yIG1heCh4K3cvMix5K2gvMikgPiAxKzFlLTY6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIGNsYXNzIG9yIGJveCBnZW9tZXRyeS4iKQogICAgICAgICAgICBpZiBsZW4oZXhwZWN0ZWQpICE9IDUgb3Igbm90IGFsbChtYXRoLmlzZmluaXRlKHYpIGZvciB2IGluIGV4cGVjdGVkKSBvciBhbnkoYWJzKGEtYik+MWUtOCBmb3IgYSxiIGluIHppcCh2YWx1ZXMsZXhwZWN0ZWQpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1hbmlmZXN0L2xhYmVsIGdlb21ldHJ5IG1pc21hdGNoLiIpCiAgICAgICAgICAgIGNsYXNzZXNbKGRvbWFpbixzcGxpdCldLmFkZChpbnQoYykpCiAgICAgICAgcGFyZW50c1tyb3dbInBhcmVudF9pZCJdXS5hZGQoc3BsaXQpCiAgICAgICAgaWYgcm93WyJncm91cF9pZCJdIGlzIG5vdCBOb25lOgogICAgICAgICAgICBncm91cHNbKGRvbWFpbixyb3dbImdyb3VwX2lkIl0pXS5hZGQoc3BsaXQpCiAgICAgICAgaGFzaGVzW3Jvd1siaW1hZ2Vfc2hhMjU2Il1dLmFkZChzcGxpdCkKICAgIGlmIGFueShsZW4odik+MSBmb3IgcG9vbCBpbiAocGFyZW50cyxncm91cHMsaGFzaGVzKSBmb3IgdiBpbiBwb29sLnZhbHVlcygpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQYXJlbnQsIGdyb3VwIG9yIGV4YWN0LWltYWdlIGxlYWthZ2UgYWNyb3NzIHNwbGl0cy4iKQogICAgZm9yIGRvbWFpbiBpbiAoInNvdXJjZSIsICJ0YXJnZXQiKToKICAgICAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiKToKICAgICAgICAgICAgaWYgY2xhc3Nlc1soZG9tYWluLHNwbGl0KV0gIT0gc2V0KHJhbmdlKDYpIGlmIGRvbWFpbiA9PSAic291cmNlIiBlbHNlIHJhbmdlKDEsNikpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1pc3NpbmcgcmVxdWlyZWQgY2xhc3Nlczoge2RvbWFpbn0ve3NwbGl0fSIpCiAgICByZXR1cm4gZGljdChDb3VudGVyKGYne3JbImRhdGFzZXQiXX0ve3JbInNwbGl0Il19L3tyWyJraW5kIl19JyBmb3IgciBpbiBtYW5pZmVzdFsidmlld3MiXSkpCgoKZGVmIHZlcmlmeV9wYWNrYWdlKHJvb3QpOgogICAgcm9vdCA9IFBhdGgocm9vdCkKICAgIHJlY29yZCA9IGpzb24ubG9hZHMoKHJvb3QvInBhY2thZ2VfaGFzaGVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGZvciBuYW1lLCBjaGVja3N1bSBpbiByZWNvcmRbInNoYTI1NiJdLml0ZW1zKCk6CiAgICAgICAgaWYgZGlnZXN0KHNhZmVfcGF0aChyb290LG5hbWUpKSAhPSBjaGVja3N1bToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBhY2thZ2UgY2hlY2tzdW0gbWlzbWF0Y2g6IHtuYW1lfSIpCiAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMoKHJvb3QvIm1hbmlmZXN0Lmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHJlcXVpcmVkID0ge3Jba10gZm9yIHIgaW4gbWFuaWZlc3RbInZpZXdzIl0gZm9yIGsgaW4gKCJpbWFnZSIsImxhYmVsIil9CiAgICByZXF1aXJlZCB8PSB7Im1hbmlmZXN0Lmpzb24iLCAid2VpZ2h0cy95b2xvdjhuLnB0IiwgIndlaWdodHMveW9sb3Y4cy5wdCIsICJ3ZWlnaHRzL3NvdXJjZV9iYXNlbGluZS5wdCIsICJ3ZWlnaHRzL3RhcmdldF9iYXNlbGluZS5wdCJ9CiAgICBpZiBub3QgcmVxdWlyZWQgPD0gc2V0KHJlY29yZFsic2hhMjU2Il0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkluY29tcGxldGUgcGFja2FnZSBoYXNoIGNvdmVyYWdlLiIpCiAgICByZWZlcmVuY2VkID0ge3Jba10gZm9yIHIgaW4gbWFuaWZlc3RbInZpZXdzIl0gZm9yIGsgaW4gKCJpbWFnZSIsICJsYWJlbCIpfQogICAgYWN0dWFsID0ge3AucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSBmb3IgZm9sZGVyIGluICgiaW1hZ2VzIiwibGFiZWxzIikKICAgICAgICAgICAgICBmb3IgcCBpbiAocm9vdC9mb2xkZXIpLnJnbG9iKCIqIikgaWYgcC5pc19maWxlKCkgYW5kIHAuc3VmZml4ICE9ICIuY2FjaGUifQogICAgaWYgYWN0dWFsICE9IHJlZmVyZW5jZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5saXN0ZWQgaW1hZ2Ugb3IgbGFiZWwgaW4gcGFja2FnZS4iKQogICAgcmV0dXJuIHZlcmlmeV92aWV3cyhyb290LG1hbmlmZXN0KQoKCmRlZiBleHRyYWN0X3BhY2thZ2UoYXJjaGl2ZSwgZGVzdGluYXRpb24pOgogICAgIiIiRXh0cmFjdCBpbnRvIGEgbmV3IGZvbGRlciBvbmx5OyByZWplY3QgZHVwbGljYXRlLCB0cmF2ZXJzYWwgYW5kIHN5bWxpbmsgZW50cmllcy4iIiIKICAgIGRlc3RpbmF0aW9uID0gUGF0aChkZXN0aW5hdGlvbikucmVzb2x2ZSgpCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVc2UgYSBuZXcgZXh0cmFjdGlvbiBkaXJlY3Rvcnk7IGV4aXN0aW5nIGRhdGEgd2lsbCBub3QgYmUgb3ZlcndyaXR0ZW4uIikKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGFyY2hpdmUpIGFzIHo6CiAgICAgICAgbmFtZXMgPSB6Lm5hbWVsaXN0KCkKICAgICAgICBpZiBsZW4obmFtZXMpICE9IGxlbihzZXQobmFtZXMpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIFpJUCBlbnRyaWVzLiIpCiAgICAgICAgZm9yIGluZm8gaW4gei5pbmZvbGlzdCgpOgogICAgICAgICAgICBzYWZlX3BhdGgoZGVzdGluYXRpb24sIGluZm8uZmlsZW5hbWUpCiAgICAgICAgICAgIGlmIChpbmZvLmV4dGVybmFsX2F0dHIgPj4gMTYpICYgMG8xNzAwMDAgPT0gMG8xMjAwMDA6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTeW1saW5rIGluIFpJUC4iKQogICAgICAgIGlmIHoudGVzdHppcCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEYW1hZ2VkIFpJUC4iKQogICAgICAgIGRlc3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAgICB6LmV4dHJhY3RhbGwoZGVzdGluYXRpb24pCiAgICB2ZXJpZnlfcGFja2FnZShkZXN0aW5hdGlvbikKICAgIHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBjb25maWd1cmUocm9vdCwgb3V0cHV0KToKICAgICIiIlNlcGFyYXRlIHNvdXJjZS90YXJnZXQgcmVjaXBlczsgdmFsIGFsd2F5cyB1c2VzIHVuY2hhbmdlZCBvcmlnaW5hbCB2aWV3cy4iIiIKICAgIGltcG9ydCB5YW1sCiAgICByb290LCBvdXRwdXQgPSBQYXRoKHJvb3QpLnJlc29sdmUoKSwgUGF0aChvdXRwdXQpLnJlc29sdmUoKQogICAgdmVyaWZ5X3BhY2thZ2Uocm9vdCkKICAgIG1hbmlmZXN0ID0ganNvbi5sb2Fkcygocm9vdC8ibWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgY29uZmlncywgY29udGVudHMgPSB7fSwge30KICAgIGZvciByZWNpcGUsIGRvbWFpbiwgZnVsbF9vbmx5IGluIFsoInNvdXJjZV9mdWxsIiwic291cmNlIixUcnVlKSwgKCJzb3VyY2Vfdmlld3MiLCJzb3VyY2UiLEZhbHNlKSwgKCJ0YXJnZXQiLCJ0YXJnZXQiLFRydWUpXToKICAgICAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiKToKICAgICAgICAgICAgaW1hZ2VzID0gW3N0cihyb290L3JbImltYWdlIl0pIGZvciByIGluIG1hbmlmZXN0WyJ2aWV3cyJdIGlmIHJbImRhdGFzZXQiXT09ZG9tYWluIGFuZCByWyJzcGxpdCJdPT1zcGxpdCBhbmQgKG5vdCBmdWxsX29ubHkgb3Igclsia2luZCJdPT0iZnVsbCIpXQogICAgICAgICAgICBjb250ZW50c1tvdXRwdXQvZiJ7cmVjaXBlfV97c3BsaXR9LnR4dCJdID0gIlxuIi5qb2luKGltYWdlcykrIlxuIgogICAgICAgIHBhdGggPSBvdXRwdXQvZiJ7cmVjaXBlfS55YW1sIgogICAgICAgIGNvbnRlbnRzW3BhdGhdID0geWFtbC5zYWZlX2R1bXAoeyJwYXRoIjpzdHIocm9vdCksICJ0cmFpbiI6c3RyKG91dHB1dC9mIntyZWNpcGV9X3RyYWluLnR4dCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsIjpzdHIob3V0cHV0L2Yie3JlY2lwZX1fdmFsLnR4dCIpLCAibmFtZXMiOk5BTUVTfSkKICAgICAgICBjb25maWdzW3JlY2lwZV0gPSBzdHIocGF0aCkKICAgIGZvciBwYXRoLCBjb250ZW50IGluIGNvbnRlbnRzLml0ZW1zKCk6CiAgICAgICAgaWYgcGF0aC5leGlzdHMoKSBhbmQgcGF0aC5yZWFkX3RleHQoKSAhPSBjb250ZW50OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeGlzdGluZyBjb25maWd1cmF0aW9uIGRpZmZlcnM6ICIgKyBzdHIocGF0aCkpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBmb3IgcGF0aCwgY29udGVudCBpbiBjb250ZW50cy5pdGVtcygpOgogICAgICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICBwYXRoLndyaXRlX3RleHQoY29udGVudCkKICAgIHJldHVybiBjb25maWdzCg==', 'sha256': '1214793aa1513c82067430e9c5abcd48d8925b3175a29fa30fd3593bb4b16589'}, 'checkpoints.py': {'base64': 'IiIiRnJvemVuIGNoZWNrcG9pbnQgaGVscGVycyBhZGFwdGVkIGZyb20gYWNjdXJhY3kvdHJhaW5fcmVjb3ZlcnkucHkuCgpQZXItcnVuIGFyY2hpdmUgbmFtZXMgYXZvaWQgY29sbGlzaW9ucyBiZXR3ZWVuIHNwZWNpYWxpc3QgZXhwZXJpbWVudHMuClVzZSBvbmx5IHRydXN0ZWQsIGxvY2FsbHkgcHJvZHVjZWQgUHlUb3JjaCBjaGVja3BvaW50cy4KIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgaGFzaGxpYixqc29uLG9zLHppcGZpbGUsY3N2LGlvLHRlbXBmaWxlCgpjbGFzcyBUcmFpbmluZ1BhdXNlZChFeGNlcHRpb24pOgogIiIiTGVhdmUgdGhlIHRyYWluaW5nIGxvb3AgYmVmb3JlIGZpbmFsX2V2YWwgc3RyaXBzIHRoZSBvcHRpbWl6ZXIuIiIiCgoKZGVmIGNoZWNrcG9pbnRfcGF0aChydW4pOgogcmV0dXJuIFBhdGgocnVuKS53aXRoX25hbWUoUGF0aChydW4pLm5hbWUgKyAnX2NoZWNrcG9pbnQuemlwJykKCgpkZWYgcmVzdG9yZV9jaGVja3BvaW50KGFyY2hpdmUscnVuLHNpZ25hdHVyZT1Ob25lKToKICIiIlZlcmlmeSB0aGUgZW50aXJlIGJhY2t1cCBiZWZvcmUgcmVwbGFjaW5nIGFueSBydW4gZmlsZXMuIE93biBjaGVja3BvaW50cyBvbmx5LiIiIgogYXJjaGl2ZT1QYXRoKGFyY2hpdmUpO3J1bj1QYXRoKHJ1bikucmVzb2x2ZSgpCiB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmNoaXZlKSBhcyB6OgogIG5hbWVzPXoubmFtZWxpc3QoKQogIGlmIGxlbihuYW1lcykhPWxlbihzZXQobmFtZXMpKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0R1cGxpY2F0ZSBjaGVja3BvaW50IGFyY2hpdmUgZW50cmllcy4nKQogIGZvciBuYW1lIGluIG5hbWVzOgogICBpZiBQYXRoKG5hbWUpLmlzX2Fic29sdXRlKCkgb3IgJ1xcJyBpbiBuYW1lIG9yICcuLicgaW4gUGF0aChuYW1lKS5wYXJ0cyBvciBub3QgKHJ1bi9uYW1lKS5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocnVuKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1Vuc2FmZSBjaGVja3BvaW50IHBhdGguJykKICBmb3IgaW5mbyBpbiB6LmluZm9saXN0KCk6CiAgIGlmICgoaW5mby5leHRlcm5hbF9hdHRyID4+IDE2KSAmIDBvMTcwMDAwKSA9PSAwbzEyMDAwMDpyYWlzZSBSdW50aW1lRXJyb3IoJ1N5bWxpbmsgaW4gY2hlY2twb2ludC4nKQogIGlmIHoudGVzdHppcCgpIGlzIG5vdCBOb25lOnJhaXNlIFJ1bnRpbWVFcnJvcignQ2hlY2twb2ludCBaSVAgaXMgZGFtYWdlZC4nKQogIHJlY29yZD1qc29uLmxvYWRzKHoucmVhZCgnc2lnbmF0dXJlLmpzb24nKSkKICBleHBlY3RlZD1zaWduYXR1cmUKICBpZiAocnVuLydzaWduYXR1cmUuanNvbicpLmV4aXN0cygpOgogICBleGlzdGluZz1qc29uLmxvYWRzKChydW4vJ3NpZ25hdHVyZS5qc29uJykucmVhZF90ZXh0KCkpWydzaGEyNTYnXQogICBpZiBleHBlY3RlZCBpcyBub3QgTm9uZSBhbmQgZXhpc3RpbmchPWV4cGVjdGVkOnJhaXNlIFJ1bnRpbWVFcnJvcignUnVuIHNpZ25hdHVyZSBtaXNtYXRjaC4nKQogICBleHBlY3RlZD1leGlzdGluZwogIGlmIGV4cGVjdGVkIGlzIG5vdCBOb25lIGFuZCByZWNvcmRbJ3NoYTI1NiddIT1leHBlY3RlZDpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgZGF0YXNldC9yZWNpcGUgbWlzbWF0Y2guJykKICBmb3IgcmVxdWlyZWQgaW4gWyd0cmFpbmluZy93ZWlnaHRzL2xhc3QucHQnLCd0cmFpbmluZy93ZWlnaHRzL2Jlc3QucHQnLCd0cmFpbmluZy9yZXN1bHRzLmNzdicsJ3RyYWluaW5nL2FyZ3MueWFtbCcsJ3RyYWluaW5nL3RyYWluZXJfc3RhdGUucHQnLCdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nXToKICAgaWYgcmVxdWlyZWQgbm90IGluIG5hbWVzOnJhaXNlIFJ1bnRpbWVFcnJvcignSW5jb21wbGV0ZSBjaGVja3BvaW50OiAnK3JlcXVpcmVkKQogIGlmICdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nIGluIG5hbWVzOgogICBtYW5pZmVzdD1qc29uLmxvYWRzKHoucmVhZCgnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJykpCiAgIGFjdHVhbD1zZXQobmFtZXMpLXsnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJ30KICAgaWYgYWN0dWFsIT1zZXQobWFuaWZlc3RbJ3NoYTI1NiddKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgZmlsZSBsaXN0IG1pc21hdGNoLicpCiAgIGZvciBuYW1lLGRpZ2VzdCBpbiBtYW5pZmVzdFsnc2hhMjU2J10uaXRlbXMoKToKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHoucmVhZChuYW1lKSkuaGV4ZGlnZXN0KCkhPWRpZ2VzdDpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgY2hlY2tzdW0gbWlzbWF0Y2g6ICcrbmFtZSkKICBydW4ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KGRpcj1ydW4ucGFyZW50LHByZWZpeD0nY2hlY2twb2ludC1yZXN0b3JlLScpIGFzIHRlbXBvcmFyeToKICAgc3RhZ2luZz1QYXRoKHRlbXBvcmFyeSkKICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICBpZiBuYW1lLmVuZHN3aXRoKCcvJyk6Y29udGludWUKICAgIHRhcmdldD1zdGFnaW5nL25hbWU7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7dGFyZ2V0LndyaXRlX2J5dGVzKHoucmVhZChuYW1lKSkKICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICBpZiBuYW1lLmVuZHN3aXRoKCcvJyk6Y29udGludWUKICAgIHRhcmdldD1ydW4vbmFtZTt0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKTtvcy5yZXBsYWNlKHN0YWdpbmcvbmFtZSx0YXJnZXQpCiBwcmludCgnVmVyaWZpZWQgY2hlY2twb2ludCByZXN0b3JlZDonLGFyY2hpdmUsZmx1c2g9VHJ1ZSkKCgpkZWYgc2F2ZV9jaGVja3BvaW50KHRyYWluZXIscnVuKToKICIiIlB1Ymxpc2ggYSBjb21wbGV0ZSwgY2hlY2tzdW1tZWQgYmFja3VwIGF0b21pY2FsbHkgYWZ0ZXIgZXZlcnkgc2F2ZWQgZXBvY2guIiIiCiBpbXBvcnQgdG9yY2gscmFuZG9tLG51bXB5IGFzIG5wCiBydW49UGF0aChydW4pO2Vwb2NoPWludCh0cmFpbmVyLmVwb2NoKQogc3RhdGU9eydlcG9jaCc6ZXBvY2gsJ3RyYWluaW5nX2ZpbmlzaGVkJzpib29sKHRyYWluZXIuc3RvcCksCiAgJ3N0b3BwZXInOntrZXk6Z2V0YXR0cih0cmFpbmVyLnN0b3BwZXIsa2V5KSBmb3Iga2V5IGluIFsnYmVzdF9maXRuZXNzJywnYmVzdF9lcG9jaCcsJ3Bvc3NpYmxlX3N0b3AnLCdwYXRpZW5jZSddfSwKICAnc2NoZWR1bGVyJzp0cmFpbmVyLnNjaGVkdWxlci5zdGF0ZV9kaWN0KCksJ3B5dGhvbl9ybmcnOnJhbmRvbS5nZXRzdGF0ZSgpLAogICdudW1weV9ybmcnOm5wLnJhbmRvbS5nZXRfc3RhdGUoKSwndG9yY2hfcm5nJzp0b3JjaC5nZXRfcm5nX3N0YXRlKCksCiAgJ2N1ZGFfcm5nJzp0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGVfYWxsKCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmV9CiBzdGF0ZV9maWxlPXJ1bi8ndHJhaW5pbmcvdHJhaW5lcl9zdGF0ZS5wdCc7c3RhdGVfZmlsZS5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiBzdGF0ZV90bXA9c3RhdGVfZmlsZS53aXRoX3N1ZmZpeCgnLnRtcCcpO3RvcmNoLnNhdmUoc3RhdGUsc3RhdGVfdG1wKTtvcy5yZXBsYWNlKHN0YXRlX3RtcCxzdGF0ZV9maWxlKQogZmlsZXM9W1BhdGgodHJhaW5lci5sYXN0KSxQYXRoKHRyYWluZXIuYmVzdCkscnVuLyd0cmFpbmluZy9hcmdzLnlhbWwnLHJ1bi8ndHJhaW5pbmcvcmVzdWx0cy5jc3YnLHJ1bi8nc2lnbmF0dXJlLmpzb24nLHN0YXRlX2ZpbGVdCiBmaWxlcy5leHRlbmQocCBmb3IgcCBpbiBbcnVuLydiYXNlbGluZV9zb3VyY2VfdmFsaWRhdGlvbi5qc29uJyxydW4vJ2Jhc2VsaW5lX3RhcmdldF92YWxpZGF0aW9uLmpzb24nXSBpZiBwLmlzX2ZpbGUoKSkKIGlmIGFueShub3QgcC5pc19maWxlKCkgZm9yIHAgaW4gZmlsZXMpOnJhaXNlIFJ1bnRpbWVFcnJvcignQ2Fubm90IHB1Ymxpc2ggYW4gaW5jb21wbGV0ZSBjaGVja3BvaW50LicpCiBhcmNoaXZlPWNoZWNrcG9pbnRfcGF0aChydW4pO3RlbXBvcmFyeT1hcmNoaXZlLndpdGhfc3VmZml4KCcudG1wJykKIHRyeToKICBoYXNoZXM9e30KICB3aXRoIHppcGZpbGUuWmlwRmlsZSh0ZW1wb3JhcnksJ3cnLHppcGZpbGUuWklQX0RFRkxBVEVELGNvbXByZXNzbGV2ZWw9MSkgYXMgejoKICAgZm9yIHAgaW4gZmlsZXM6CiAgICBwYXlsb2FkPXAucmVhZF9ieXRlcygpO25hbWU9cC5yZWxhdGl2ZV90byhydW4pLmFzX3Bvc2l4KCk7aGFzaGVzW25hbWVdPWhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpO3oud3JpdGVzdHIobmFtZSxwYXlsb2FkKQogICB6LndyaXRlc3RyKCdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nLGpzb24uZHVtcHMoeydmb3JtYXQnOidwY2JfcGF1c2VfcmVzdW1lX3YxJywnY29tcGxldGVkX2Vwb2Nocyc6ZXBvY2grMSwndHJhaW5pbmdfZmluaXNoZWQnOnN0YXRlWyd0cmFpbmluZ19maW5pc2hlZCddLCdzaGEyNTYnOmhhc2hlc30pKQogIHdpdGggemlwZmlsZS5aaXBGaWxlKHRlbXBvcmFyeSkgYXMgejoKICAgaWYgei50ZXN0emlwKCkgaXMgbm90IE5vbmU6cmFpc2UgUnVudGltZUVycm9yKCdDaGVja3BvaW50IHZlcmlmaWNhdGlvbiBmYWlsZWQuJykKICBvcy5yZXBsYWNlKHRlbXBvcmFyeSxhcmNoaXZlKQogZmluYWxseTp0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKIHByaW50KGYnQ0hFQ0tQT0lOVCBTQVZFRDogZXBvY2gge2Vwb2NoKzF9IOKAlCB7YXJjaGl2ZX0nLGZsdXNoPVRydWUpCiByZXR1cm4gYXJjaGl2ZQoKCmRlZiByZXN0b3JlX3RyYWluZXJfc3RhdGUodHJhaW5lcixydW4pOgogaW1wb3J0IHRvcmNoLHJhbmRvbSxudW1weSBhcyBucAogcGF0aD1QYXRoKHJ1bikvJ3RyYWluaW5nL3RyYWluZXJfc3RhdGUucHQnCiBpZiBub3QgcGF0aC5leGlzdHMoKSBvciBub3QgdHJhaW5lci5yZXN1bWU6cmV0dXJuCiBzdGF0ZT10b3JjaC5sb2FkKHBhdGgsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSkKIGlmIHN0YXRlWydlcG9jaCddKzEhPXRyYWluZXIuc3RhcnRfZXBvY2g6cmFpc2UgUnVudGltZUVycm9yKCdDaGVja3BvaW50IGFuZCB0cmFpbmVyIHN0YXRlIHJlZmVyIHRvIGRpZmZlcmVudCBlcG9jaHMuJykKIGZvciBrZXksdmFsdWUgaW4gc3RhdGVbJ3N0b3BwZXInXS5pdGVtcygpOnNldGF0dHIodHJhaW5lci5zdG9wcGVyLGtleSx2YWx1ZSkKIHRyYWluZXIuc2NoZWR1bGVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnc2NoZWR1bGVyJ10pCiByYW5kb20uc2V0c3RhdGUoc3RhdGVbJ3B5dGhvbl9ybmcnXSk7bnAucmFuZG9tLnNldF9zdGF0ZShzdGF0ZVsnbnVtcHlfcm5nJ10pO3RvcmNoLnNldF9ybmdfc3RhdGUoc3RhdGVbJ3RvcmNoX3JuZyddKQogaWYgc3RhdGVbJ2N1ZGFfcm5nJ10gaXMgbm90IE5vbmUgYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6dG9yY2guY3VkYS5zZXRfcm5nX3N0YXRlX2FsbChzdGF0ZVsnY3VkYV9ybmcnXSkKCgpkZWYgdHJpbV9yZXN1bHRzKHJ1bixjb21wbGV0ZWRfZXBvY2hzKToKICIiIkRpc2NhcmQgQ1NWIHJvd3Mgd3JpdHRlbiBieSBhbiBlcG9jaCB3aG9zZSB3ZWlnaHRzIHdlcmUgbm90IGNvbW1pdHRlZC4iIiIKIHBhdGg9UGF0aChydW4pLyd0cmFpbmluZy9yZXN1bHRzLmNzdicKIGlmIG5vdCBwYXRoLmV4aXN0cygpOnJldHVybgogd2l0aCBwYXRoLm9wZW4obmV3bGluZT0nJykgYXMgc3RyZWFtOgogIHJlYWRlcj1jc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKTtmaWVsZHM9cmVhZGVyLmZpZWxkbmFtZXMKICByb3dzPVtyb3cgZm9yIHJvdyBpbiByZWFkZXIgaWYgaW50KGZsb2F0KHJvd1snZXBvY2gnXSkpPD1jb21wbGV0ZWRfZXBvY2hzXQogb3V0cHV0PWlvLlN0cmluZ0lPKCk7d3JpdGVyPWNzdi5EaWN0V3JpdGVyKG91dHB1dCxmaWVsZG5hbWVzPWZpZWxkcyk7d3JpdGVyLndyaXRlaGVhZGVyKCk7d3JpdGVyLndyaXRlcm93cyhyb3dzKQogcGF0aC53cml0ZV90ZXh0KG91dHB1dC5nZXR2YWx1ZSgpKQoK', 'sha256': 'd235ebf57d1d4581e9cc38d52ea6fe727ff333f0b620ffb73286769cc9dbc419'}, 'smoke.py': {'base64': 'IiIiVGlueSB0cmFpbmluZy1zZXQgbWVtb3JpemF0aW9uIGFuZCBmcmVzaC1wcm9jZXNzIHJlc3VtZSBjaGVjaywgTk9UIGV2YWx1YXRpb24uCgpVc2VzIHNpeCB0cmFpbmluZyBjcm9wcywgb25lIHBlciBjbGFzcy4gTm8gdmFsaWRhdGlvbiBvciB0ZXN0IGltYWdlcyBhcmUgdXNlZC4KVGhlIGRlbGliZXJhdGVseSByZXBlYXRlZCB0cmFpbi9ldmFsIHNhbXBsZSB0ZXN0cyB3aGV0aGVyIHRoZSBwaXBlbGluZSBjYW4gbGVhcm4uCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKZnJvbSByZXZhbXAucnVudGltZSBpbXBvcnQgZGlnZXN0LCB2ZXJpZnlfcGFja2FnZQoKUkVDSVBFID0gZGljdChlcG9jaHM9ODAsIHBhdGllbmNlPTEwMCwgaW1nc3o9MzIwLCBiYXRjaD02LCB3b3JrZXJzPTAsCiAgICAgICAgICAgICAgb3B0aW1pemVyPSJBZGFtVyIsIGxyMD0uMDAzLCBscmY9LjEsIG5icz02LCBjb3NfbHI9VHJ1ZSwKICAgICAgICAgICAgICB3YXJtdXBfZXBvY2hzPTAsIHNlZWQ9NDIsIGRldGVybWluaXN0aWM9VHJ1ZSwgYW1wPUZhbHNlLAogICAgICAgICAgICAgIG1vc2FpYz0wLCBtaXh1cD0wLCBjb3B5X3Bhc3RlPTAsIGRlZ3JlZXM9MCwgdHJhbnNsYXRlPTAsIHNjYWxlPTAsCiAgICAgICAgICAgICAgZmxpcGxyPTAsIGZsaXB1ZD0wLCBoc3ZfaD0wLCBoc3Zfcz0wLCBoc3Zfdj0wLAogICAgICAgICAgICAgIGNsb3NlX21vc2FpYz0wLCBzYXZlPVRydWUsIHNhdmVfcGVyaW9kPS0xLCBwbG90cz1GYWxzZSkKCgpkZWYgc2lnbmF0dXJlKHBhY2thZ2UpOgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdWx0cmFseXRpY3MKICAgIHJldHVybiBoYXNobGliLnNoYTI1NigoZGlnZXN0KFBhdGgocGFja2FnZSkvInBhY2thZ2VfaGFzaGVzLmpzb24iKSArIGpzb24uZHVtcHMoUkVDSVBFLCBzb3J0X2tleXM9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgKyB0b3JjaC5fX3ZlcnNpb25fXyArIHVsdHJhbHl0aWNzLl9fdmVyc2lvbl9fKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgd29ya2VyKHBhY2thZ2UsIGRhdGEsIHJ1biwgcmVzdW1lLCBkZXZpY2UpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBmcm9tIHJldmFtcCBpbXBvcnQgY2hlY2twb2ludHMgYXMgY2sKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcyg0KQogICAgcnVuID0gUGF0aChydW4pCiAgICBleHBlY3RlZCA9IHNpZ25hdHVyZShwYWNrYWdlKQogICAgaWYgcmVzdW1lOgogICAgICAgIGlmIGpzb24ubG9hZHMoKHJ1bi8ic2lnbmF0dXJlLmpzb24iKS5yZWFkX3RleHQoKSlbInNoYTI1NiJdICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGVja3BvaW50IHNpZ25hdHVyZSBkaWZmZXJzIGZyb20gZGF0YXNldC9yZWNpcGUvZW52aXJvbm1lbnQuIikKICAgICAgICByYXcgPSB0b3JjaC5sb2FkKHJ1bi8idHJhaW5pbmcvd2VpZ2h0cy9sYXN0LnB0IiwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgY29tcGxldGVkID0gaW50KHJhd1siZXBvY2giXSkgKyAxCiAgICAgICAgaWYgY29tcGxldGVkIDwgMSBvciByYXdbIm9wdGltaXplciJdIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGEgY29tcGxldGVkIGVwb2NoIHdpdGggcmVzdW1hYmxlIG9wdGltaXplciBzdGF0ZS4iKQogICAgICAgIGRlbCByYXcKICAgICAgICBjay50cmltX3Jlc3VsdHMocnVuLCBjb21wbGV0ZWQpCiAgICAgICAgaWYgY29tcGxldGVkID49IFJFQ0lQRVsiZXBvY2hzIl06CiAgICAgICAgICAgIHByaW50KCJBbGwgZXBvY2hzIGFscmVhZHkgc2F2ZWQ7IHJlYnVpbGRpbmcgdGhlIHJlcG9ydCB3aXRob3V0IHJldHJhaW5pbmcuIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmV0dXJuCiAgICBlbHNlOgogICAgICAgIHJ1bi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgICAgIChydW4vInNpZ25hdHVyZS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHsic2hhMjU2IjpleHBlY3RlZCwgInJlY2lwZSI6UkVDSVBFfSkpCiAgICBtb2RlbCA9IFlPTE8oc3RyKHJ1bi8idHJhaW5pbmcvd2VpZ2h0cy9sYXN0LnB0IiBpZiByZXN1bWUgZWxzZSBQYXRoKHBhY2thZ2UpLyJ3ZWlnaHRzL3lvbG92OG4ucHQiKSkKICAgIGRlZiBzYXZlKHRyYWluZXIpOgogICAgICAgIGNrLnNhdmVfY2hlY2twb2ludCh0cmFpbmVyLCBydW4pCiAgICBkZWYgcGF1c2UodHJhaW5lcik6CiAgICAgICAgaWYgbm90IHJlc3VtZSBhbmQgdHJhaW5lci5lcG9jaCA9PSAwOgogICAgICAgICAgICByYWlzZSBjay5UcmFpbmluZ1BhdXNlZCgpCiAgICBkZWYgcmVzdG9yZWQodHJhaW5lcik6CiAgICAgICAgY2sucmVzdG9yZV90cmFpbmVyX3N0YXRlKHRyYWluZXIscnVuKQogICAgICAgIGlmIHJlc3VtZToKICAgICAgICAgICAgaWYgdHJhaW5lci5zdGFydF9lcG9jaCAhPSBjb21wbGV0ZWQgb3Igbm90IHRyYWluZXIub3B0aW1pemVyLnN0YXRlOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJPcHRpbWl6ZXIvZXBvY2ggd2FzIG5vdCByZXN0b3JlZC4iKQogICAgICAgICAgICAocnVuLyJyZXN1bWVfdmVyaWZpZWQuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7InN0YXJ0X2Vwb2NoIjp0cmFpbmVyLnN0YXJ0X2Vwb2NoLAogICAgICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZV9lbnRyaWVzIjpsZW4odHJhaW5lci5vcHRpbWl6ZXIuc3RhdGUpLCAicGlkIjpvcy5nZXRwaWQoKX0pKQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCJvbl9tb2RlbF9zYXZlIixzYXZlKQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCJvbl9maXRfZXBvY2hfZW5kIixwYXVzZSkKICAgIG1vZGVsLmFkZF9jYWxsYmFjaygib25fdHJhaW5fc3RhcnQiLHJlc3RvcmVkKQogICAgdHJ5OgogICAgICAgIGlmIHJlc3VtZToKICAgICAgICAgICAgbW9kZWwudHJhaW4ocmVzdW1lPVRydWUsIGRldmljZT1kZXZpY2UsIGRhdGE9c3RyKGRhdGEpLCBzYXZlX2Rpcj1zdHIocnVuLyJ0cmFpbmluZyIpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG1vZGVsLnRyYWluKGRhdGE9c3RyKGRhdGEpLCBkZXZpY2U9ZGV2aWNlLCBwcm9qZWN0PXN0cihydW4pLCBuYW1lPSJ0cmFpbmluZyIsIGV4aXN0X29rPVRydWUsICoqUkVDSVBFKQogICAgZXhjZXB0IGNrLlRyYWluaW5nUGF1c2VkOgogICAgICAgIHJldHVybgogICAgZmluYWxseToKICAgICAgICB0cmFpbmVyID0gZ2V0YXR0cihtb2RlbCwidHJhaW5lciIsTm9uZSkKICAgICAgICBmb3IgbG9hZGVyIGluIFtnZXRhdHRyKHRyYWluZXIsInRyYWluX2xvYWRlciIsTm9uZSksZ2V0YXR0cih0cmFpbmVyLCJ0ZXN0X2xvYWRlciIsTm9uZSldOgogICAgICAgICAgICBpZiBjYWxsYWJsZShnZXRhdHRyKGxvYWRlciwiY2xvc2UiLE5vbmUpKToKICAgICAgICAgICAgICAgIGxvYWRlci5jbG9zZSgpCgoKZGVmIGNoZWNrKHBhY2thZ2UsIG91dHB1dCwgZGV2aWNlPSJjcHUiLCBjaGVja3BvaW50PU5vbmUsIHJlc3VtZV9vbmx5PUZhbHNlKToKICAgICIiIlJlc3RhcnQtc2FmZSBjb29yZGluYXRvci4gVGhlIGxhc3QgYXRvbWljYWxseSBwdWJsaXNoZWQgWklQIGlzIGF1dGhvcml0YXRpdmUuIiIiCiAgICBpbXBvcnQgZmNudGwKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0KS5yZXNvbHZlKCkKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIChvdXRwdXQvInJ1bi5sb2NrIikub3BlbigiYSIpIGFzIGxvY2s6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmY250bC5mbG9jayhsb2NrLCBmY250bC5MT0NLX0VYIHwgZmNudGwuTE9DS19OQikKICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoaXMgcnVuIGlzIGFscmVhZHkgYWN0aXZlOyBkbyBub3QgbGF1bmNoIGEgc2Vjb25kIHdvcmtlci4iKSBmcm9tIGVycm9yCiAgICAgICAgcmV0dXJuIF9jaGVjayhwYWNrYWdlLCBvdXRwdXQsIGRldmljZSwgY2hlY2twb2ludCwgcmVzdW1lX29ubHkpCgoKZGVmIGNoZWNrcG9pbnRfYXJjaGl2ZShwYXRoLCBvdXRwdXQpOgogICAgIiIiQWNjZXB0IGEgZG93bmxvYWRlZCBaSVAgb3IgYSBLYWdnbGUtZXhwYW5kZWQgY2hlY2twb2ludCBkaXJlY3RvcnkuIiIiCiAgICBpbXBvcnQgemlwZmlsZQogICAgZnJvbSByZXZhbXAucnVudGltZSBpbXBvcnQgc2FmZV9wYXRoCiAgICBwYXRoID0gUGF0aChwYXRoKS5yZXNvbHZlKCkKICAgIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLnN1ZmZpeCA9PSAiLnppcCI6CiAgICAgICAgcmV0dXJuIHBhdGgKICAgIGlmIHBhdGgubmFtZSA9PSAiY2hlY2twb2ludF9tYW5pZmVzdC5qc29uIjoKICAgICAgICBwYXRoID0gcGF0aC5wYXJlbnQKICAgIHJlY29yZCA9IGpzb24ubG9hZHMoKHBhdGgvImNoZWNrcG9pbnRfbWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgZm9yIG5hbWUsIGV4cGVjdGVkIGluIHJlY29yZFsic2hhMjU2Il0uaXRlbXMoKToKICAgICAgICBpZiBkaWdlc3Qoc2FmZV9wYXRoKHBhdGgsbmFtZSkpICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGFuZ2VkIGV4cGFuZGVkIGNoZWNrcG9pbnQ6ICIgKyBuYW1lKQogICAgdGFyZ2V0ID0gUGF0aChvdXRwdXQpLyJpbXBvcnRlZF9jaGVja3BvaW50LnppcCIKICAgIHRlbXBvcmFyeSA9IHRhcmdldC53aXRoX3N1ZmZpeCgiLnRtcCIpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh0ZW1wb3JhcnksInciLHppcGZpbGUuWklQX0RFRkxBVEVELGNvbXByZXNzbGV2ZWw9MSkgYXMgejoKICAgICAgICBmb3IgbmFtZSBpbiBbKnJlY29yZFsic2hhMjU2Il0sImNoZWNrcG9pbnRfbWFuaWZlc3QuanNvbiJdOgogICAgICAgICAgICB6LndyaXRlKHNhZmVfcGF0aChwYXRoLG5hbWUpLG5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZSh0YXJnZXQpCiAgICByZXR1cm4gdGFyZ2V0CgoKZGVmIF9jaGVjayhwYWNrYWdlLCBvdXRwdXQsIGRldmljZSwgY2hlY2twb2ludCwgcmVzdW1lX29ubHkpOgogICAgaW1wb3J0IHlhbWwKICAgIGZyb20gcmV2YW1wIGltcG9ydCBjaGVja3BvaW50cyBhcyBjawogICAgcGFja2FnZSwgb3V0cHV0ID0gUGF0aChwYWNrYWdlKS5yZXNvbHZlKCksIFBhdGgob3V0cHV0KS5yZXNvbHZlKCkKICAgIHZlcmlmeV9wYWNrYWdlKHBhY2thZ2UpCiAgICBleHBlY3RlZCA9IHNpZ25hdHVyZShwYWNrYWdlKQogICAgbWFya2VyID0gb3V0cHV0LyJydW5fc2lnbmF0dXJlLmpzb24iCiAgICBpZiBtYXJrZXIuZXhpc3RzKCkgYW5kIGpzb24ubG9hZHMobWFya2VyLnJlYWRfdGV4dCgpKVsic2hhMjU2Il0gIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhpc3RpbmcgcnVuIHVzZXMgYW5vdGhlciBwYWNrYWdlLCByZWNpcGUgb3IgZW52aXJvbm1lbnQuIikKICAgIGlmIChvdXRwdXQvInJlcG9ydC5qc29uIikuZXhpc3RzKCk6CiAgICAgICAgaWYgbm90IG1hcmtlci5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29tcGxldGVkIHJ1biBsYWNrcyBhIG1hdGNoaW5nIHJlc3VtZSBtYXJrZXI7IHVzZSBhIG5ldyBvdXRwdXQgZGlyZWN0b3J5LiIpCiAgICAgICAgcmVwb3J0ID0ganNvbi5sb2Fkcygob3V0cHV0LyJyZXBvcnQuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHJlcG9ydFsicGFja2FnZV9oYXNoZXNfc2hhMjU2Il0gIT0gZGlnZXN0KHBhY2thZ2UvInBhY2thZ2VfaGFzaGVzLmpzb24iKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVwb3J0L3BhY2thZ2UgbWlzbWF0Y2guIikKICAgICAgICBpZiByZXBvcnRbInN0YXR1cyJdICE9ICJwYXNzZWQiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvbXBsZXRlZCBsZWFybmluZyBjaGVjayBmYWlsZWQ7IGluc3BlY3QgaXRzIHJlcG9ydCByYXRoZXIgdGhhbiBzaWxlbnRseSByZXRyYWluaW5nLiIpCiAgICAgICAgcHJpbnQoIkFscmVhZHkgY29tcGxldGU7IHJldXNpbmcgcmVwb3J0IHdpdGhvdXQgdHJhaW5pbmcuIiwgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gcmVwb3J0CiAgICBmaXJzdCwgZnJlc2ggPSBvdXRwdXQvImZpcnN0X3J1biIsIG91dHB1dC8iZnJlc2hfcnVuIgogICAgbG9jYWwgPSBjay5jaGVja3BvaW50X3BhdGgoZnJlc2gpIGlmIGNrLmNoZWNrcG9pbnRfcGF0aChmcmVzaCkuZXhpc3RzKCkgZWxzZSBjay5jaGVja3BvaW50X3BhdGgoZmlyc3QpCiAgICBpZiBjaGVja3BvaW50IGFuZCBsb2NhbC5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJMb2NhbCBwcm9ncmVzcyBleGlzdHMuIENsZWFyIFJFU1RPUkVfRlJPTSB0byByZXN1bWUgaXQ7IHVzZSBhIG5ldyBydW4gZm9sZGVyIGZvciBhbiBpbXBvcnRlZCBjaGVja3BvaW50LiIpCiAgICBpZiByZXN1bWVfb25seSBhbmQgbm90IGNoZWNrcG9pbnQgYW5kIG5vdCBsb2NhbC5leGlzdHMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZSByZXF1ZXN0ZWQgYnV0IG5vIGNoZWNrcG9pbnQgZXhpc3RzLiBObyB0cmFpbmluZyB3YXMgc3RhcnRlZC4iKQogICAgaWYgY2hlY2twb2ludDoKICAgICAgICBpbXBvcnRlZCA9IGNoZWNrcG9pbnRfYXJjaGl2ZShjaGVja3BvaW50LCBvdXRwdXQpCiAgICAgICAgY2sucmVzdG9yZV9jaGVja3BvaW50KGltcG9ydGVkLCBmcmVzaCwgZXhwZWN0ZWQpCiAgICAgICAgIyBQdWJsaXNoIHRoZSB2ZXJpZmllZCBpbXBvcnRlZCBhcmNoaXZlIGxvY2FsbHkgYmVmb3JlIGxhdW5jaGluZyBhbnkgd29yay4KICAgICAgICBzaHV0aWwuY29weWZpbGUoaW1wb3J0ZWQsIGNrLmNoZWNrcG9pbnRfcGF0aChmcmVzaCkpCiAgICAgICAgbG9jYWwgPSBjay5jaGVja3BvaW50X3BhdGgoZnJlc2gpCiAgICBtYXJrZXIud3JpdGVfdGV4dChqc29uLmR1bXBzKHsic2hhMjU2IjpleHBlY3RlZCwicmVjaXBlIjpSRUNJUEV9KSkKICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcygocGFja2FnZS8ibWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgc2VsZWN0ZWQgPSB7fQogICAgZm9yIHJvdyBpbiBtYW5pZmVzdFsidmlld3MiXToKICAgICAgICBpZiByb3dbImRhdGFzZXQiXT09InNvdXJjZSIgYW5kIHJvd1sic3BsaXQiXT09InRyYWluIiBhbmQgcm93WyJraW5kIl09PSJjcm9wIjoKICAgICAgICAgICAgY2xhc3NlcyA9IHtpbnQoYm94WzBdKSBmb3IgYm94IGluIHJvd1sibGFiZWxzIl19CiAgICAgICAgICAgIGlmIGxlbihjbGFzc2VzKT09MToKICAgICAgICAgICAgICAgIHNlbGVjdGVkLnNldGRlZmF1bHQobmV4dChpdGVyKGNsYXNzZXMpKSxyb3cpCiAgICBpZiBzZXQoc2VsZWN0ZWQpICE9IHNldChyYW5nZSg2KSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2Fubm90IGZvcm0gYSBzaXgtY2xhc3MgbGVhcm5pbmcgY2hlY2sgZnJvbSB0cmFpbmluZyBjcm9wcy4iKQogICAgZGF0YSA9IG91dHB1dC8idGlueSI7IChkYXRhLyJpbWFnZXMiKS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7IChkYXRhLyJsYWJlbHMiKS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZm9yIGNscywgcm93IGluIHNvcnRlZChzZWxlY3RlZC5pdGVtcygpKToKICAgICAgICBmb3Iga2V5LCBkZXN0aW5hdGlvbiBpbiBbKCJpbWFnZSIsZGF0YS9mImltYWdlcy97Y2xzfS5qcGciKSwoImxhYmVsIixkYXRhL2YibGFiZWxzL3tjbHN9LnR4dCIpXToKICAgICAgICAgICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBpZiBkaWdlc3QoZGVzdGluYXRpb24pICE9IHJvd1trZXkrIl9zaGEyNTYiXToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGFuZ2VkIHRpbnkgbGVhcm5pbmcgaW5wdXQ6ICIgKyBzdHIoZGVzdGluYXRpb24pKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlKHBhY2thZ2Uvcm93W2tleV0sZGVzdGluYXRpb24pCiAgICBjZmcgPSBkYXRhLyJkYXRhLnlhbWwiCiAgICBjZmcud3JpdGVfdGV4dCh5YW1sLnNhZmVfZHVtcCh7InBhdGgiOnN0cihkYXRhKSwgInRyYWluIjoiaW1hZ2VzIiwgInZhbCI6ImltYWdlcyIsICJuYW1lcyI6bWFuaWZlc3RbIm5hbWVzIl19KSkKICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbiwgWU9MT19DT05GSUdfRElSPXN0cihvdXRwdXQvInlvbG9fc2V0dGluZ3MiKSwgV0FOREJfTU9ERT0iZGlzYWJsZWQiLCBDT01FVF9NT0RFPSJESVNBQkxFRCIpCiAgICBjb21tYW5kID0gW3N5cy5leGVjdXRhYmxlLCItbSIsInJldmFtcC5zbW9rZSIsIi0tcGFja2FnZSIsc3RyKHBhY2thZ2UpLCItLXdvcmtlciIsIi0tZGF0YSIsc3RyKGNmZyksIi0tZGV2aWNlIixzdHIoZGV2aWNlKV0KICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgaWYgbm90IGxvY2FsLmV4aXN0cygpOgogICAgICAgIGlmIGZpcnN0LmV4aXN0cygpOgogICAgICAgICAgICAjIEFuIGludGVycnVwdGlvbiBiZWZvcmUgdGhlIGZpcnN0IGNoZWNrcG9pbnQgaGFzIG5vIGVwb2NoIHRvIHJlc3VtZS4KICAgICAgICAgICAgIyBQcmVzZXJ2ZSB0aGF0IHBhcnRpYWwgZGlyZWN0b3J5IGluc3RlYWQgb2Ygb3ZlcndyaXRpbmcgaXRzIGZpbGVzLgogICAgICAgICAgICBmaXJzdC5yZW5hbWUob3V0cHV0L2YiaW5jb21wbGV0ZV9maXJzdF9ydW5fe3RpbWUudGltZV9ucygpfSIpCiAgICAgICAgcHJpbnQoIlN0YXJ0aW5nIHRpbnkgY2hlY2s7IHRoZSBmaXJzdCBjaGVja3BvaW50IHdpbGwgYmUgc2F2ZWQgYWZ0ZXIgZXBvY2ggMS4iLCBmbHVzaD1UcnVlKQogICAgICAgIHdpdGggKG91dHB1dC8icGF1c2UubG9nIikub3BlbigiYSIpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3VicHJvY2Vzcy5ydW4oY29tbWFuZCtbIi0tb3V0cHV0IixzdHIoZmlyc3QpXSwgZW52PWVudixzdGRvdXQ9c3RyZWFtLHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCxjaGVjaz1UcnVlKQogICAgICAgIGxvY2FsID0gY2suY2hlY2twb2ludF9wYXRoKGZpcnN0KQogICAgY2sucmVzdG9yZV9jaGVja3BvaW50KGxvY2FsLGZyZXNoLGV4cGVjdGVkKQogICAgaWYgbG9jYWwgIT0gY2suY2hlY2twb2ludF9wYXRoKGZyZXNoKToKICAgICAgICBzaHV0aWwuY29weWZpbGUobG9jYWwsY2suY2hlY2twb2ludF9wYXRoKGZyZXNoKSkKICAgIGlmIGZpcnN0LmV4aXN0cygpOgogICAgICAgIGZpcnN0LnJlbmFtZShvdXRwdXQvZiJwcmVzZXJ2ZWRfZmlyc3RfcnVuX3t0aW1lLnRpbWVfbnMoKX0iKQogICAgcHJpbnQoIkNvbnRpbnVpbmcgZnJvbSB0aGUgbGFzdCBjb21wbGV0ZSBjaGVja3BvaW50LiBQcm9ncmVzczogIiArIHN0cihvdXRwdXQvInJlc3VtZS5sb2ciKSxmbHVzaD1UcnVlKQogICAgd2l0aCAob3V0cHV0LyJyZXN1bWUubG9nIikub3BlbigiYSIpIGFzIHN0cmVhbToKICAgICAgICBzdWJwcm9jZXNzLnJ1bihjb21tYW5kK1siLS1vdXRwdXQiLHN0cihmcmVzaCksIi0tcmVzdW1lIl0sZW52PWVudixzdGRvdXQ9c3RyZWFtLHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCxjaGVjaz1UcnVlKQogICAgd2l0aCAoZnJlc2gvInRyYWluaW5nL3Jlc3VsdHMuY3N2Iikub3BlbigpIGFzIHN0cmVhbToKICAgICAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKSkKICAgIGVwb2NocyA9IFtpbnQoZmxvYXQocm93WyJlcG9jaCJdKSkgZm9yIHJvdyBpbiByb3dzXQogICAgaWYgZXBvY2hzICE9IGxpc3QocmFuZ2UoMSxSRUNJUEVbImVwb2NocyJdKzEpKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZWQgQ1NWIGhhcyBtaXNzaW5nIG9yIHJlcGVhdGVkIGVwb2Nocy4iKQogICAgZmlyc3RfbG9zcyA9IHN1bShmbG9hdChyWyJ0cmFpbi9ib3hfbG9zcyJdKSBmb3IgciBpbiByb3dzWzo1XSkvNQogICAgbGFzdF9sb3NzID0gc3VtKGZsb2F0KHJbInRyYWluL2JveF9sb3NzIl0pIGZvciByIGluIHJvd3NbLTU6XSkvNQogICAgYXA1MCA9IG1heChmbG9hdChyWyJtZXRyaWNzL21BUDUwKEIpIl0pIGZvciByIGluIHJvd3MpCiAgICBsZWFybmVkID0gbGFzdF9sb3NzIDwgLjg1KmZpcnN0X2xvc3MgYW5kIGFwNTAgPj0gLjUKICAgIHJlcG9ydCA9IHsic3RhdHVzIjoicGFzc2VkIiBpZiBsZWFybmVkIGVsc2UgImZhaWxlZF9sZWFybmluZ19nYXRlIiwgInNlY29uZHMiOnJvdW5kKHRpbWUubW9ub3RvbmljKCktc3RhcnQsMSksCiAgICAgICAgICAgICAgInNjb3BlIjoiU2l4IFRSQUlOSU5HIGNyb3BzIGV2YWx1YXRlZCBvbiB0aGVtc2VsdmVzLiBGdW5jdGlvbmFsIGV2aWRlbmNlIE9OTFk7IG5vdCBtb2RlbCBhY2N1cmFjeS4iLAogICAgICAgICAgICAgICJyZWNpcGUiOlJFQ0lQRSwgInBhY2thZ2VfaGFzaGVzX3NoYTI1NiI6ZGlnZXN0KHBhY2thZ2UvInBhY2thZ2VfaGFzaGVzLmpzb24iKSwKICAgICAgICAgICAgICAic2VsZWN0ZWRfcGFyZW50X2lkcyI6W3JbInBhcmVudF9pZCJdIGZvciByIGluIHNlbGVjdGVkLnZhbHVlcygpXSwKICAgICAgICAgICAgICAiY2hlY2tzIjpbIlJlYWwgdHJhaW5pbmcgcGF1c2VkIGFmdGVyIG9uZSBjb21wbGV0ZWQgZXBvY2giLCAiT3B0aW1pemVyIHJldGFpbmVkIGluIGNoZWNrc3VtbWVkIFpJUCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJSZXN1bWUgaW4gc2VwYXJhdGUgUHl0aG9uIHByb2Nlc3MgYW5kIGZyZXNoIGZvbGRlciB3aXRoIG9yaWdpbmFsIHJ1biBtb3ZlZCBhd2F5IiwKICAgICAgICAgICAgICAgICAgICAgICAgIk9wdGltaXplciByZXN0b3JlZCBhdCBzYXZlZCBlcG9jaCAob3IgZmluYWwgY2hlY2twb2ludCByZXVzZWQpIiwgIkNTViBoYXMgZWFjaCBwbGFubmVkIGVwb2NoIGV4YWN0bHkgb25jZSJdLAogICAgICAgICAgICAgICJsZWFybmluZ19nYXRlIjp7ImZpcnN0XzVfYm94X2xvc3MiOmZpcnN0X2xvc3MsImxhc3RfNV9ib3hfbG9zcyI6bGFzdF9sb3NzLCJiZXN0X3RyYWluX3NhbXBsZV9hcDUwIjphcDUwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJlcXVpcmVkIjoiTGFzdC1maXZlIG1lYW4gYm94IGxvc3MgPDg1JSBvZiBmaXJzdC1maXZlOyBtZW1vcml6ZWQtc2FtcGxlIEFQNTAgPj0wLjUwIn19CiAgICByZXBvcnRfdG1wID0gb3V0cHV0LyJyZXBvcnQudG1wIgogICAgcmVwb3J0X3RtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yKSsiXG4iKQogICAgcmVwb3J0X3RtcC5yZXBsYWNlKG91dHB1dC8icmVwb3J0Lmpzb24iKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIpKQogICAgaWYgbm90IGxlYXJuZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaW55IGxlYXJuaW5nIGNoZWNrIGZhaWxlZC4gSW5zcGVjdCBsb2dzOyBkbyBub3Qgc3RhcnQgZnVsbCB0cmFpbmluZy4iKQogICAgcmV0dXJuIHJlcG9ydAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wYWNrYWdlIix0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0Iix0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIixkZWZhdWx0PSJjcHUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrZXIiLGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VtZSIsYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YSIsdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50Iix0eXBlPVBhdGgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VtZS1vbmx5IixhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGlmIGFyZ3Mud29ya2VyOgogICAgICAgIHdvcmtlcihhcmdzLnBhY2thZ2UsYXJncy5kYXRhLGFyZ3Mub3V0cHV0LGFyZ3MucmVzdW1lLGFyZ3MuZGV2aWNlKQogICAgZWxzZToKICAgICAgICBjaGVjayhhcmdzLnBhY2thZ2UsYXJncy5vdXRwdXQsYXJncy5kZXZpY2UsYXJncy5jaGVja3BvaW50LGFyZ3MucmVzdW1lX29ubHkpCg==', 'sha256': '4579b39a6afba6191efa1599716bcdd9667aba987becad80c04e6a8d11ff65ae'}, 'train_full_boards.py': {'base64': 'IiIiU291cmNlIGZ1bGwtYm9hcmQgYWJsYXRpb246IGlkZW50aWNhbCByZWNpcGUsIGRlcml2ZWQgdHJhaW5pbmcgdmlld3MgZXhjbHVkZWQuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgc3RhdGlzdGljcwppbXBvcnQgdGltZQoKZnJvbSByZXZhbXAgaW1wb3J0IGNoZWNrcG9pbnRzIGFzIGNrCmZyb20gcmV2YW1wLnJ1bnRpbWUgaW1wb3J0IE5BTUVTLCBjb25maWd1cmUsIGRpZ2VzdCwgdmVyaWZ5X3BhY2thZ2UKZnJvbSByZXZhbXAuc21va2UgaW1wb3J0IGNoZWNrcG9pbnRfYXJjaGl2ZQoKCmRlZiByZWNpcGUoZG9tYWluKToKICAgICMgS2VlcCB0aGUgdmVyaWZpZWQgcGlsb3QgZ2VvbWV0cnkvcHJlY2lzaW9uOyB1c2UgYSBmdWxsIHRyYWluaW5nIHNjaGVkdWxlLgogICAgaWYgZG9tYWluICE9ICJzb3VyY2UiOiByYWlzZSBWYWx1ZUVycm9yKCJUaGlzIGV4cGVyaW1lbnQgdHJhaW5zIHRoZSBzb3VyY2Ugc3BlY2lhbGlzdCBvbmx5LiIpCiAgICByZXR1cm4gZGljdChlcG9jaHM9NjAsIHBhdGllbmNlPTE1LCBpbWdzej05NjAgaWYgZG9tYWluID09ICdzb3VyY2UnIGVsc2UgNjQwLAogICAgICAgICAgICAgICAgYmF0Y2g9NCBpZiBkb21haW4gPT0gJ3NvdXJjZScgZWxzZSA4LCB3b3JrZXJzPTAsIG9wdGltaXplcj0nQWRhbVcnLAogICAgICAgICAgICAgICAgbHIwPS4wMDA1LCBscmY9LjEsIG5icz00IGlmIGRvbWFpbiA9PSAnc291cmNlJyBlbHNlIDgsCiAgICAgICAgICAgICAgICBjb3NfbHI9VHJ1ZSwgd2FybXVwX2Vwb2Nocz0zLCB3YXJtdXBfYmlhc19scj0wLiwgc2VlZD00MiwgZGV0ZXJtaW5pc3RpYz1UcnVlLAogICAgICAgICAgICAgICAgYW1wPUZhbHNlLCBtb3NhaWM9MC4sIG1peHVwPTAuLCBjb3B5X3Bhc3RlPTAuLCBjdXRtaXg9MC4sCiAgICAgICAgICAgICAgICBkZWdyZWVzPTAuLCB0cmFuc2xhdGU9MC4sIHNjYWxlPTAuLCBzaGVhcj0wLiwgcGVyc3BlY3RpdmU9MC4sCiAgICAgICAgICAgICAgICBmbGlwbHI9MC4sIGZsaXB1ZD0wLiwgaHN2X2g9MC4sIGhzdl9zPTAuLCBoc3Zfdj0wLiwgYmdyPTAuLAogICAgICAgICAgICAgICAgYXVnbWVudGF0aW9ucz1bXSwgY2xvc2VfbW9zYWljPTAsIHNhdmU9VHJ1ZSwgc2F2ZV9wZXJpb2Q9LTEsCiAgICAgICAgICAgICAgICBwbG90cz1GYWxzZSwgY2FjaGU9RmFsc2UpCgoKZGVmIGVudmlyb25tZW50KCk6CiAgICBpbXBvcnQgcGxhdGZvcm0KICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHRvcmNodmlzaW9uCiAgICBpbXBvcnQgdWx0cmFseXRpY3MKICAgIGltcG9ydCBudW1weQogICAgcmV0dXJuIGRpY3QocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fLAogICAgICAgICAgICAgICAgdG9yY2h2aXNpb249dG9yY2h2aXNpb24uX192ZXJzaW9uX18sIHVsdHJhbHl0aWNzPXVsdHJhbHl0aWNzLl9fdmVyc2lvbl9fLAogICAgICAgICAgICAgICAgbnVtcHk9bnVtcHkuX192ZXJzaW9uX18sIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhKQoKCmRlZiBpZGVudGl0eShwYWNrYWdlLCBkb21haW4pOgogICAgcmVjb3JkID0gZGljdChwYWNrYWdlX3NoYTI1Nj1kaWdlc3QoUGF0aChwYWNrYWdlKS8ncGFja2FnZV9oYXNoZXMuanNvbicpLAogICAgICAgICAgICAgICAgICBkb21haW49ZG9tYWluLCB0cmFpbmluZ192aWV3cz0nZnVsbF9vbmx5JywgcmVjaXBlPXJlY2lwZShkb21haW4pLCBpbml0aWFsaXphdGlvbj0nd2VpZ2h0cy95b2xvdjhzLnB0JywKICAgICAgICAgICAgICAgICAgZW52aXJvbm1lbnQ9ZW52aXJvbm1lbnQoKSwgY29kZT17bmFtZTpkaWdlc3QoUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKG5hbWUpKQogICAgICAgICAgICAgICAgICAgIGZvciBuYW1lIGluIFsndHJhaW5fZnVsbF9ib2FyZHMucHknLCdjaGVja3BvaW50cy5weScsJ3J1bnRpbWUucHknLCdzbW9rZS5weSddfSkKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKHJlY29yZCxzb3J0X2tleXM9VHJ1ZSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpLCByZWNvcmQKCgpkZWYgYXRvbWljX2pzb24ocGF0aCwgcmVjb3JkKToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICB0bXAgPSBwYXRoLndpdGhfc3VmZml4KCcudG1wJykKICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVjb3JkLGluZGVudD0yLGFsbG93X25hbj1GYWxzZSkrJ1xuJykKICAgIHRtcC5yZXBsYWNlKHBhdGgpCgoKZGVmIG1ldHJpY3NfcmVjb3JkKHJlc3VsdCk6CiAgICBib3ggPSByZXN1bHQuYm94CiAgICByZXR1cm4gZGljdChtYXA1MD1mbG9hdChib3gubWFwNTApLCBtYXA1MF85NT1mbG9hdChib3gubWFwKSwKICAgICAgICAgICAgICAgIHByZWNpc2lvbj1mbG9hdChib3gubXApLCByZWNhbGw9ZmxvYXQoYm94Lm1yKSwKICAgICAgICAgICAgICAgIHBlcl9jbGFzcz17TkFNRVNbaW50KGMpXTpkaWN0KGFwNTA9ZmxvYXQoYm94LmFwNTBbaV0pLAogICAgICAgICAgICAgICAgICAgIGFwNTBfOTU9ZmxvYXQoYm94LmFwW2ldKSwgcHJlY2lzaW9uPWZsb2F0KGJveC5wW2ldKSwgcmVjYWxsPWZsb2F0KGJveC5yW2ldKSkKICAgICAgICAgICAgICAgICAgICBmb3IgaSxjIGluIGVudW1lcmF0ZShib3guYXBfY2xhc3NfaW5kZXgpfSwKICAgICAgICAgICAgICAgIHNwZWVkX21zX3Blcl9pbWFnZT17azpmbG9hdCh2KSBmb3Igayx2IGluIHJlc3VsdC5zcGVlZC5pdGVtcygpfSkKCgpkZWYgZXZhbHVhdGUod2VpZ2h0cywgZGF0YSwgcnVuLCBuYW1lLCBzZXR0aW5ncywgZGV2aWNlLCBleHBlY3RlZCk6CiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBwYXRoID0gcnVuLyhuYW1lKycuanNvbicpCiAgICBrZXkgPSBkaWN0KHJ1bl9zaWduYXR1cmU9ZXhwZWN0ZWQsIHdlaWdodHNfc2hhMjU2PWRpZ2VzdCh3ZWlnaHRzKSwgaW1nc3o9c2V0dGluZ3NbJ2ltZ3N6J10sCiAgICAgICAgICAgICAgIGJhdGNoPXNldHRpbmdzWydiYXRjaCddLCBjb25mPS4wMDEsIGlvdT0uNywgbWF4X2RldD0zMDAsIHNwbGl0PSd2YWwnKQogICAgaWYgcGF0aC5leGlzdHMoKToKICAgICAgICByZWNvcmQgPSBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KCkpCiAgICAgICAgaWYgcmVjb3JkWydpZGVudGl0eSddICE9IGtleToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU2F2ZWQgZXZhbHVhdGlvbiBkaWZmZXJzOiAnK3N0cihwYXRoKSkKICAgICAgICByZXR1cm4gcmVjb3JkCiAgICBzdGFydD10aW1lLm1vbm90b25pYygpCiAgICBtb2RlbD1ZT0xPKHN0cih3ZWlnaHRzKSkKICAgIGlmIGxpc3QobW9kZWwubmFtZXMudmFsdWVzKCkpICE9IE5BTUVTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0Jhc2VsaW5lL2NhbmRpZGF0ZSBjbGFzcyBvcmRlciBkaWZmZXJzLicpCiAgICByZXN1bHQ9bW9kZWwudmFsKGRhdGE9c3RyKGRhdGEpLHNwbGl0PSd2YWwnLGltZ3N6PXNldHRpbmdzWydpbWdzeiddLGJhdGNoPXNldHRpbmdzWydiYXRjaCddLAogICAgICAgICAgICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLHdvcmtlcnM9MCxoYWxmPUZhbHNlLGNvbmY9LjAwMSxpb3U9LjcsbWF4X2RldD0zMDAsCiAgICAgICAgICAgICAgICAgICAgIGF1Z21lbnQ9RmFsc2UscGxvdHM9RmFsc2Usc2F2ZV9qc29uPUZhbHNlLHByb2plY3Q9c3RyKHJ1bi8nZXZhbHVhdGlvbicpLAogICAgICAgICAgICAgICAgICAgICBuYW1lPW5hbWUsZXhpc3Rfb2s9VHJ1ZSx2ZXJib3NlPVRydWUpCiAgICByZWNvcmQ9ZGljdChpZGVudGl0eT1rZXksbWV0cmljcz1tZXRyaWNzX3JlY29yZChyZXN1bHQpLHNlY29uZHM9dGltZS5tb25vdG9uaWMoKS1zdGFydCkKICAgIGF0b21pY19qc29uKHBhdGgscmVjb3JkKQogICAgcmV0dXJuIHJlY29yZAoKCmRlZiBhc3NlcnRfbm9fYWxidW1lbnRhdGlvbnMoZGF0YXNldCk6CiAgICAiIiJGYWlsIGlmIHRoZSBpbnN0YWxsZWQgdHJhaW5lciBzaWxlbnRseSBlbmFibGVzIGl0cyBvcHRpb25hbCBkZWZhdWx0IHRyYW5zZm9ybXMuIiIiCiAgICBkZWYgdmlzaXQodHJhbnNmb3JtKToKICAgICAgICBpZiB0eXBlKHRyYW5zZm9ybSkuX19uYW1lX18gPT0gJ0FsYnVtZW50YXRpb25zJzoKICAgICAgICAgICAgYXBwbGllZD1nZXRhdHRyKHRyYW5zZm9ybSwndHJhbnNmb3JtJyxOb25lKQogICAgICAgICAgICBpZiBhcHBsaWVkIGlzIG5vdCBOb25lIGFuZCBnZXRhdHRyKGFwcGxpZWQsJ3RyYW5zZm9ybXMnLFtdKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignVW5leHBlY3RlZCBBbGJ1bWVudGF0aW9ucyB0cmFuc2Zvcm1zIGVuYWJsZWQuJykKICAgICAgICBmb3IgY2hpbGQgaW4gZ2V0YXR0cih0cmFuc2Zvcm0sJ3RyYW5zZm9ybXMnLFtdKToKICAgICAgICAgICAgdmlzaXQoY2hpbGQpCiAgICB2aXNpdChkYXRhc2V0LnRyYW5zZm9ybXMpCgoKZGVmIHRyYWluKHBhY2thZ2UsIGRhdGEsIHJ1biwgZG9tYWluLCBkZXZpY2UsIGV4cGVjdGVkLCByZXN1bWUpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBzZXR0aW5ncz1yZWNpcGUoZG9tYWluKQogICAgY29tcGxldGVkPTAKICAgIGlmIHJlc3VtZToKICAgICAgICByYXc9dG9yY2gubG9hZChydW4vJ3RyYWluaW5nL3dlaWdodHMvbGFzdC5wdCcsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBjb21wbGV0ZWQ9aW50KHJhd1snZXBvY2gnXSkrMQogICAgICAgIGlmIGNvbXBsZXRlZCA8IDEgb3IgcmF3WydvcHRpbWl6ZXInXSBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IGhhcyBubyBjb21wbGV0ZWQgZXBvY2gvb3B0aW1pemVyLicpCiAgICAgICAgZGVsIHJhdwogICAgICAgIGNrLnRyaW1fcmVzdWx0cyhydW4sY29tcGxldGVkKQogICAgICAgIHN0YXRlPXRvcmNoLmxvYWQocnVuLyd0cmFpbmluZy90cmFpbmVyX3N0YXRlLnB0JyxtYXBfbG9jYXRpb249J2NwdScsd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIGlmIHN0YXRlWydlcG9jaCddKzEgIT0gY29tcGxldGVkOiByYWlzZSBWYWx1ZUVycm9yKCdUcmFpbmVyL2NoZWNrcG9pbnQgZXBvY2ggbWlzbWF0Y2guJykKICAgICAgICBpZiBjb21wbGV0ZWQgPj0gc2V0dGluZ3NbJ2Vwb2NocyddIG9yIHN0YXRlWyd0cmFpbmluZ19maW5pc2hlZCddOgogICAgICAgICAgICBwcmludCgnVHJhaW5pbmcgYWxyZWFkeSBmaW5pc2hlZDsgZXZhbHVhdGlvbi9yZXBvcnQgcmVjb3Zlcnkgb25seS4nLGZsdXNoPVRydWUpCiAgICAgICAgICAgIHJldHVybgogICAgbW9kZWw9WU9MTyhzdHIocnVuLyd0cmFpbmluZy93ZWlnaHRzL2xhc3QucHQnIGlmIHJlc3VtZSBlbHNlIHBhY2thZ2UvJ3dlaWdodHMveW9sb3Y4cy5wdCcpKQogICAgc3RhcnRlZD1bTm9uZV0KICAgIGRlZiBvbl9zdGFydCh0cmFpbmVyKToKICAgICAgICBhc3NlcnRfbm9fYWxidW1lbnRhdGlvbnModHJhaW5lci50cmFpbl9sb2FkZXIuZGF0YXNldCkKICAgICAgICBjay5yZXN0b3JlX3RyYWluZXJfc3RhdGUodHJhaW5lcixydW4pCiAgICAgICAgaWYgcmVzdW1lIGFuZCAodHJhaW5lci5zdGFydF9lcG9jaCAhPSBjb21wbGV0ZWQgb3Igbm90IHRyYWluZXIub3B0aW1pemVyLnN0YXRlKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPcHRpbWl6ZXIvZXBvY2ggcmVjb3ZlcnkgZmFpbGVkLicpCiAgICAgICAgcHJpbnQoZidTT1VSQ0UgVFJBSU5JTkcgUkVBRFk6IGVwb2NoIHt0cmFpbmVyLnN0YXJ0X2Vwb2NoKzF9OyBvcHRpb25hbCBhdWdtZW50YXRpb25zIGRpc2FibGVkJyxmbHVzaD1UcnVlKQogICAgZGVmIGVwb2NoX3N0YXJ0KHRyYWluZXIpOgogICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6dG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICAgICAgc3RhcnRlZFswXT10aW1lLm1vbm90b25pYygpCiAgICBkZWYgc2F2ZSh0cmFpbmVyKToKICAgICAgICByZWNvcmQ9anNvbi5sb2FkcygocnVuLydzaWduYXR1cmUuanNvbicpLnJlYWRfdGV4dCgpKQogICAgICAgIHJlY29yZC5zZXRkZWZhdWx0KCdlcG9jaF9tZWFzdXJlbWVudHMnLFtdKS5hcHBlbmQoZGljdChlcG9jaD10cmFpbmVyLmVwb2NoKzEsCiAgICAgICAgICAgIHRyYWluX2FuZF92YWxfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWRbMF0sCiAgICAgICAgICAgIHBlYWtfYWxsb2NhdGVkX2dpYj10b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMTAyNCoqMyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZSwKICAgICAgICAgICAgcGVha19yZXNlcnZlZF9naWI9dG9yY2guY3VkYS5tYXhfbWVtb3J5X3Jlc2VydmVkKCkvMTAyNCoqMyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZSkpCiAgICAgICAgYXRvbWljX2pzb24ocnVuLydzaWduYXR1cmUuanNvbicscmVjb3JkKQogICAgICAgIGNrLnNhdmVfY2hlY2twb2ludCh0cmFpbmVyLHJ1bikKICAgIG1vZGVsLmFkZF9jYWxsYmFjaygnb25fdHJhaW5fc3RhcnQnLG9uX3N0YXJ0KQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCdvbl90cmFpbl9lcG9jaF9zdGFydCcsZXBvY2hfc3RhcnQpCiAgICBtb2RlbC5hZGRfY2FsbGJhY2soJ29uX21vZGVsX3NhdmUnLHNhdmUpCiAgICB0cnk6CiAgICAgICAgaWYgcmVzdW1lOgogICAgICAgICAgICBtb2RlbC50cmFpbihyZXN1bWU9VHJ1ZSxkZXZpY2U9ZGV2aWNlLGRhdGE9c3RyKGRhdGEpLHNhdmVfZGlyPXN0cihydW4vJ3RyYWluaW5nJyksYXVnbWVudGF0aW9ucz1bXSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBtb2RlbC50cmFpbihkYXRhPXN0cihkYXRhKSxkZXZpY2U9ZGV2aWNlLHByb2plY3Q9c3RyKHJ1biksbmFtZT0ndHJhaW5pbmcnLGV4aXN0X29rPVRydWUsKipzZXR0aW5ncykKICAgIGZpbmFsbHk6CiAgICAgICAgdHJhaW5lcj1nZXRhdHRyKG1vZGVsLCd0cmFpbmVyJyxOb25lKQogICAgICAgIGZvciBsb2FkZXIgaW4gW2dldGF0dHIodHJhaW5lciwndHJhaW5fbG9hZGVyJyxOb25lKSxnZXRhdHRyKHRyYWluZXIsJ3Rlc3RfbG9hZGVyJyxOb25lKV06CiAgICAgICAgICAgIGlmIGNhbGxhYmxlKGdldGF0dHIobG9hZGVyLCdjbG9zZScsTm9uZSkpOmxvYWRlci5jbG9zZSgpCgoKZGVmIHJ1bihwYWNrYWdlLCBvdXRwdXQsIGRvbWFpbj0nc291cmNlJywgZGV2aWNlPScwJywgY2hlY2twb2ludD1Ob25lLCByZXN1bWVfb25seT1GYWxzZSk6CiAgICBpbXBvcnQgZmNudGwKICAgIHBhY2thZ2Usb3V0cHV0PVBhdGgocGFja2FnZSkucmVzb2x2ZSgpLFBhdGgob3V0cHV0KS5yZXNvbHZlKCkKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggKG91dHB1dC8ncnVuLmxvY2snKS5vcGVuKCdhJykgYXMgbG9jazoKICAgICAgICB0cnk6ZmNudGwuZmxvY2sobG9jayxmY250bC5MT0NLX0VYfGZjbnRsLkxPQ0tfTkIpCiAgICAgICAgZXhjZXB0IEJsb2NraW5nSU9FcnJvciBhcyBlcnJvcjpyYWlzZSBSdW50aW1lRXJyb3IoJ1NvdXJjZSB0cmFpbmluZyBhbHJlYWR5IGFjdGl2ZS4nKSBmcm9tIGVycm9yCiAgICAgICAgcmV0dXJuIF9ydW4ocGFja2FnZSxvdXRwdXQsZG9tYWluLGRldmljZSxjaGVja3BvaW50LHJlc3VtZV9vbmx5KQoKCmRlZiBfcnVuKHBhY2thZ2Usb3V0cHV0LGRvbWFpbixkZXZpY2UsY2hlY2twb2ludCxyZXN1bWVfb25seSk6CiAgICBpbXBvcnQgdG9yY2gKICAgIGltcG9ydCB1bHRyYWx5dGljcwogICAgaWYgdWx0cmFseXRpY3MuX192ZXJzaW9uX18gIT0gJzguNC4xNDInOnJhaXNlIFZhbHVlRXJyb3IoJ1VzZSBwaW5uZWQgVWx0cmFseXRpY3MgOC40LjE0Mi4nKQogICAgaWYgZGV2aWNlICE9ICdjcHUnIGFuZCBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0VuYWJsZSBLYWdnbGUgR1BVIGJlZm9yZSB0cmFpbmluZy4nKQogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDQpCiAgICB2ZXJpZnlfcGFja2FnZShwYWNrYWdlKQogICAgZXhwZWN0ZWQsZGV0YWlscz1pZGVudGl0eShwYWNrYWdlLGRvbWFpbikKICAgIHJ1bj1vdXRwdXQvJ2Z1bGxfYm9hcmRzX3J1bic7YXJjaGl2ZT1jay5jaGVja3BvaW50X3BhdGgocnVuKQogICAgaWYgY2hlY2twb2ludCBhbmQgYXJjaGl2ZS5leGlzdHMoKTpyYWlzZSBWYWx1ZUVycm9yKCdMb2NhbCBwcm9ncmVzcyBleGlzdHM7IGNsZWFyIFJFU1RPUkVfRlJPTS4nKQogICAgaWYgcmVzdW1lX29ubHkgYW5kIG5vdCBjaGVja3BvaW50IGFuZCBub3QgYXJjaGl2ZS5leGlzdHMoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1Jlc3VtZSByZXF1ZXN0ZWQgYnV0IG5vIGNoZWNrcG9pbnQgZXhpc3RzLicpCiAgICBtYXJrZXI9cnVuLydzaWduYXR1cmUuanNvbicKICAgIGlmIG1hcmtlci5leGlzdHMoKSBhbmQganNvbi5sb2FkcyhtYXJrZXIucmVhZF90ZXh0KCkpWydzaGEyNTYnXSAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdEaWZmZXJlbnQgcGFja2FnZSwgcmVjaXBlLCBjb2RlIG9yIGVudmlyb25tZW50OyB1c2UgYSBuZXcgb3V0cHV0IGZvbGRlci4nKQogICAgaWYgY2hlY2twb2ludDoKICAgICAgICBpbXBvcnRlZD1jaGVja3BvaW50X2FyY2hpdmUoY2hlY2twb2ludCxvdXRwdXQpCiAgICAgICAgY2sucmVzdG9yZV9jaGVja3BvaW50KGltcG9ydGVkLHJ1bixleHBlY3RlZCkKICAgICAgICBzaHV0aWwuY29weWZpbGUoaW1wb3J0ZWQsYXJjaGl2ZSkKICAgIGlmIGFyY2hpdmUuZXhpc3RzKCk6Y2sucmVzdG9yZV9jaGVja3BvaW50KGFyY2hpdmUscnVuLGV4cGVjdGVkKQogICAgaWYgbm90IG1hcmtlci5leGlzdHMoKToKICAgICAgICBydW4ubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAgICAgYXRvbWljX2pzb24obWFya2VyLGRpY3Qoc2hhMjU2PWV4cGVjdGVkLGlkZW50aXR5PWRldGFpbHMsZXBvY2hfbWVhc3VyZW1lbnRzPVtdKSkKICAgIHJlcG9ydF9wYXRoPW91dHB1dC8ncmVwb3J0Lmpzb24nCiAgICBpZiByZXBvcnRfcGF0aC5leGlzdHMoKToKICAgICAgICByZXBvcnQ9anNvbi5sb2FkcyhyZXBvcnRfcGF0aC5yZWFkX3RleHQoKSkKICAgICAgICBpZiByZXBvcnRbJ3NpZ25hdHVyZSddICE9IGV4cGVjdGVkIG9yIG5vdCBhcmNoaXZlLmV4aXN0cygpIG9yIHJlcG9ydFsnY2hlY2twb2ludF9zaGEyNTYnXSAhPSBkaWdlc3QoYXJjaGl2ZSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0NvbXBsZXRlZCByZXBvcnQvY2hlY2twb2ludCBtaXNtYXRjaC4nKQogICAgICAgIHByaW50KCdTb3VyY2UgdHJhaW5pbmcgYWxyZWFkeSBjb21wbGV0ZTsgcmV1c2luZyByZXBvcnQuJyxmbHVzaD1UcnVlKQogICAgICAgIHJldHVybiByZXBvcnQKICAgIGNvbmZpZ3M9Y29uZmlndXJlKHBhY2thZ2Usb3V0cHV0Lydjb25maWdzJykKICAgIGRhdGE9Y29uZmlnc1snc291cmNlX2Z1bGwnXQogICAgc2V0dGluZ3M9cmVjaXBlKGRvbWFpbikKICAgIGJhc2VsaW5lPWV2YWx1YXRlKHBhY2thZ2UvZid3ZWlnaHRzL3tkb21haW59X2Jhc2VsaW5lLnB0JyxkYXRhLHJ1biwKICAgICAgICAgICAgICAgICAgICAgIGYnYmFzZWxpbmVfe2RvbWFpbn1fdmFsaWRhdGlvbicsc2V0dGluZ3MsZGV2aWNlLGV4cGVjdGVkKQogICAgIyBJbnRlcnJ1cHRlZCBlcG9jaCBmaWxlcyBjYW5ub3Qgb3ZlcnJpZGUgdGhlIGF1dGhvcml0YXRpdmUgYXRvbWljIFpJUC4KICAgIGlmIG5vdCBhcmNoaXZlLmV4aXN0cygpIGFuZCAocnVuLyd0cmFpbmluZycpLmV4aXN0cygpOgogICAgICAgIChydW4vJ3RyYWluaW5nJykucmVuYW1lKHJ1bi9mJ2luY29tcGxldGVfdHJhaW5pbmdfe3RpbWUudGltZV9ucygpfScpCiAgICB0cmFpbihwYWNrYWdlLGRhdGEscnVuLGRvbWFpbixkZXZpY2UsZXhwZWN0ZWQsYXJjaGl2ZS5leGlzdHMoKSkKICAgIGNrLnJlc3RvcmVfY2hlY2twb2ludChhcmNoaXZlLHJ1bixleHBlY3RlZCkKICAgIHdpdGggKHJ1bi8ndHJhaW5pbmcvcmVzdWx0cy5jc3YnKS5vcGVuKCkgYXMgc3RyZWFtOgogICAgICAgIHJvd3M9bGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKSkKICAgIGZpbmlzaGVkPXRvcmNoLmxvYWQocnVuLyd0cmFpbmluZy90cmFpbmVyX3N0YXRlLnB0JyxtYXBfbG9jYXRpb249J2NwdScsd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgY29tcGxldGVkPXZhbGlkYXRlX2hpc3Rvcnkocm93cyxmaW5pc2hlZCxzZXR0aW5nc1snZXBvY2hzJ10pCiAgICBjYW5kaWRhdGU9ZXZhbHVhdGUocnVuLyd0cmFpbmluZy93ZWlnaHRzL2Jlc3QucHQnLGRhdGEscnVuLCdjYW5kaWRhdGVfdmFsaWRhdGlvbicsc2V0dGluZ3MsZGV2aWNlLGV4cGVjdGVkKQogICAgbWVhc3VyZW1lbnRzPWpzb24ubG9hZHMobWFya2VyLnJlYWRfdGV4dCgpKVsnZXBvY2hfbWVhc3VyZW1lbnRzJ10KICAgIGlmIFtyWydlcG9jaCddIGZvciByIGluIG1lYXN1cmVtZW50c10gIT0gbGlzdChyYW5nZSgxLGNvbXBsZXRlZCsxKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdUcmFpbmluZyB0aW1pbmcgaGlzdG9yeSBpbmNvbXBsZXRlIG9yIGR1cGxpY2F0ZWQuJykKICAgIG1hbmlmZXN0PWpzb24ubG9hZHMoKHBhY2thZ2UvJ21hbmlmZXN0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGRvbWFpbl9yb3dzPVtyIGZvciByIGluIG1hbmlmZXN0Wyd2aWV3cyddIGlmIHJbJ2RhdGFzZXQnXT09ZG9tYWluIGFuZCByWydraW5kJ109PSdmdWxsJ10KICAgIHN0ZWFkeT1zdGF0aXN0aWNzLm1lZGlhbihtWyd0cmFpbl9hbmRfdmFsX3NlY29uZHMnXSBmb3IgbSBpbiBtZWFzdXJlbWVudHNbMTpdIG9yIG1lYXN1cmVtZW50cykKICAgIHJlcG9ydD1kaWN0KHN0YXR1cz0ndHJhaW5pbmdfY29tcGxldGVkJyxjb21wbGV0ZWRfZXBvY2hzPWNvbXBsZXRlZCwKICAgICAgICAgICAgICAgIHN0b3BwZWRfZWFybHk9Y29tcGxldGVkPHNldHRpbmdzWydlcG9jaHMnXSwKICAgICAgICAgICAgICAgIHNlbGVjdGlvbl9tZXRyaWM9J3ZhbGlkYXRpb24gbUFQNTAtOTUgKFVsdHJhbHl0aWNzIDguNC4xNDIgZml0bmVzcyknLAogICAgICAgICAgICAgICAgcmV2aWV3PXJldmlld19jYW5kaWRhdGUoYmFzZWxpbmVbJ21ldHJpY3MnXSxjYW5kaWRhdGVbJ21ldHJpY3MnXSksc2lnbmF0dXJlPWV4cGVjdGVkLGNoZWNrcG9pbnRfc2hhMjU2PWRpZ2VzdChhcmNoaXZlKSwKICAgICAgICAgICAgICAgIGlkZW50aXR5PWRldGFpbHMsZGV2aWNlPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIGlmIGRldmljZSE9J2NwdScgZWxzZSAnY3B1JywKICAgICAgICAgICAgICAgIGNvdW50cz17czpzdW0oclsnc3BsaXQnXT09cyBmb3IgciBpbiBkb21haW5fcm93cykgZm9yIHMgaW4gWyd0cmFpbicsJ3ZhbCddfSwKICAgICAgICAgICAgICAgIGJhc2VsaW5lPWJhc2VsaW5lLGNhbmRpZGF0ZT1jYW5kaWRhdGUsCiAgICAgICAgICAgICAgICBkZWx0YT17azpjYW5kaWRhdGVbJ21ldHJpY3MnXVtrXS1iYXNlbGluZVsnbWV0cmljcyddW2tdIGZvciBrIGluIFsnbWFwNTAnLCdtYXA1MF85NScsJ3ByZWNpc2lvbicsJ3JlY2FsbCddfSwKICAgICAgICAgICAgICAgIGVwb2NoX21lYXN1cmVtZW50cz1tZWFzdXJlbWVudHMsc3RlYWR5X2Vwb2NoX3NlY29uZHNfZXhjbHVkaW5nX2NoZWNrcG9pbnQ9c3RlYWR5LAogICAgICAgICAgICAgICAgbG93ZXJfYm91bmRfNjBfZXBvY2hfaG91cnM9c3RlYWR5KjYwLzM2MDAsCiAgICAgICAgICAgICAgICBsaW1pdGF0aW9ucz1bJ0Z1bGwtYm9hcmQtb25seSBhYmxhdGlvbjsgY29tcGFyZSB3aXRoIGV4cGVyaW1lbnQgMS4gVmFsaWRhdGlvbiBpcyB1c2VkIGZvciBtb2RlbCBzZWxlY3Rpb24uJywKICAgICAgICAgICAgICAgICAgICAnRXN0aW1hdGUgZXhjbHVkZXMgY2hlY2twb2ludCBaSVAgd3JpdGluZywgc2V0dXAsIGZpbmFsIGV2YWx1YXRpb24gYW5kIHBvc3NpYmxlIGxhdGVyIHJlY2lwZSBjaGFuZ2VzLicsCiAgICAgICAgICAgICAgICAgICAgJ1ZhbGlkYXRpb24gaXMgc2VwYXJhdGUgZnJvbSB0cmFpbmluZzsgbm8gZGV2ZWxvcG1lbnQtdGVzdCBpbWFnZXMgdXNlZC4nLAogICAgICAgICAgICAgICAgICAgICdUYXJnZXQgbWlzc2luZy1ob2xlIHBlcmZvcm1hbmNlIHVuc3VwcG9ydGVkOyB0YXJnZXQgYm9hcmQgaW5kZXBlbmRlbmNlIHVua25vd24uJywKICAgICAgICAgICAgICAgICAgICAnTm8gbm9ybWFsIGZ1bGwtYm9hcmQgYmVuY2htYXJrOyBubyBwcm9kdWN0aW9uIG1vZGVsIHJlcGxhY2VkLiddKQogICAgYXRvbWljX2pzb24ocmVwb3J0X3BhdGgscmVwb3J0KQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIpLGZsdXNoPVRydWUpCiAgICByZXR1cm4gcmVwb3J0CgoKZGVmIHZhbGlkYXRlX2hpc3Rvcnkocm93cywgc3RhdGUsIG1heGltdW0pOgogICAgY29tcGxldGVkPWludChzdGF0ZVsnZXBvY2gnXSkrMQogICAgaWYgbm90IHN0YXRlWyd0cmFpbmluZ19maW5pc2hlZCddIG9yIG5vdCAxIDw9IGNvbXBsZXRlZCA8PSBtYXhpbXVtOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignVHJhaW5pbmcgZGlkIG5vdCByZWFjaCBhbiBpbnRlbnRpb25hbCBjb21wbGV0aW9uLicpCiAgICBpZiBbaW50KGZsb2F0KHJbJ2Vwb2NoJ10pKSBmb3IgciBpbiByb3dzXSAhPSBsaXN0KHJhbmdlKDEsY29tcGxldGVkKzEpKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1RyYWluaW5nIGVwb2NocyBpbmNvbXBsZXRlIG9yIGR1cGxpY2F0ZWQuJykKICAgIHJldHVybiBjb21wbGV0ZWQKCgpkZWYgcmV2aWV3X2NhbmRpZGF0ZShiYXNlbGluZSwgY2FuZGlkYXRlKToKICAgICMgUHJlZGVjbGFyZWQgcmV2aWV3IHRocmVzaG9sZHMsIG5vdCBzdGF0aXN0aWNhbCBzaWduaWZpY2FuY2Ugb3IgcHJvbW90aW9uLgogICAgcmVncmVzc2lvbnM9e25hbWU6Y2FuZGlkYXRlWydwZXJfY2xhc3MnXVtuYW1lXVsnYXA1MCddLXZhbHVlc1snYXA1MCddCiAgICAgICAgICAgICAgICAgZm9yIG5hbWUsdmFsdWVzIGluIGJhc2VsaW5lWydwZXJfY2xhc3MnXS5pdGVtcygpCiAgICAgICAgICAgICAgICAgaWYgbmFtZSBpbiBjYW5kaWRhdGVbJ3Blcl9jbGFzcyddIGFuZAogICAgICAgICAgICAgICAgICAgIGNhbmRpZGF0ZVsncGVyX2NsYXNzJ11bbmFtZV1bJ2FwNTAnXSA8IHZhbHVlc1snYXA1MCddLS4wNX0KICAgIGNvbXBsZXRlPXNldChjYW5kaWRhdGVbJ3Blcl9jbGFzcyddKT09c2V0KE5BTUVTKT09c2V0KGJhc2VsaW5lWydwZXJfY2xhc3MnXSkKICAgIHBhc3NlZD0oY29tcGxldGUgYW5kIGNhbmRpZGF0ZVsnbWFwNTBfOTUnXSA+PSBiYXNlbGluZVsnbWFwNTBfOTUnXSsuMDEKICAgICAgICAgICAgYW5kIGNhbmRpZGF0ZVsnbWFwNTAnXSA+PSBiYXNlbGluZVsnbWFwNTAnXS0uMDEgYW5kIG5vdCByZWdyZXNzaW9ucykKICAgIHJldHVybiBkaWN0KHdvcnRoX2Z1cnRoZXJfcmV2aWV3PXBhc3NlZCxhbGxfc2l4X2NsYXNzZXNfZXZhbHVhdGVkPWNvbXBsZXRlLAogICAgICAgICAgICAgICAgbGFyZ2VfY2xhc3NfYXA1MF9yZWdyZXNzaW9ucz1yZWdyZXNzaW9ucyxhdXRvbWF0aWNfcHJvbW90aW9uPUZhbHNlLAogICAgICAgICAgICAgICAgcnVsZT0nbUFQNTAtOTUgaW1wcm92ZXMgYnkgYXQgbGVhc3QgMSBwZXJjZW50YWdlIHBvaW50OyBtQVA1MCBkcm9wcyBubyBtb3JlIHRoYW4gMSBwb2ludDsgbm8gY2xhc3MgQVA1MCBkcm9wcyBtb3JlIHRoYW4gNSBwb2ludHMuIFJldmlldyBvbmx5OyBubyBzdGF0aXN0aWNhbCBvciBkZXBsb3ltZW50IGNsYWltLicpCgoKZGVmIG1haW4oKToKICAgIHBhcnNlcj1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1wYWNrYWdlJyx0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tb3V0cHV0Jyx0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tZG9tYWluJyxjaG9pY2VzPVsnc291cmNlJ10sZGVmYXVsdD0nc291cmNlJykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tZGV2aWNlJyxkZWZhdWx0PScwJykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcsdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1yZXN1bWUtb25seScsYWN0aW9uPSdzdG9yZV90cnVlJykKICAgIGFyZ3M9cGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCdZT0xPX0NPTkZJR19ESVInLHN0cihhcmdzLm91dHB1dC8neW9sb19zZXR0aW5ncycpKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCdXQU5EQl9NT0RFJywnZGlzYWJsZWQnKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCdDT01FVF9NT0RFJywnRElTQUJMRUQnKQogICAgcnVuKGFyZ3MucGFja2FnZSxhcmdzLm91dHB1dCxhcmdzLmRvbWFpbixhcmdzLmRldmljZSxhcmdzLmNoZWNrcG9pbnQsYXJncy5yZXN1bWVfb25seSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'sha256': 'ba92a42c459b3499b5cfe6c806e47dcaf299db3b46054dc27f8a07c9233488ce'}, 'diagnose.py': {'base64': 'IiIiVmFsaWRhdGlvbi1vbmx5IGVycm9yIGNvdW50cyBhdCBhIGZpeGVkIHRocmVzaG9sZDsgbm8gdHJhaW5pbmcgb3IgdGVzdCBhY2Nlc3MuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCB6aXBmaWxlCmZyb20gcmV2YW1wLnJ1bnRpbWUgaW1wb3J0IGRpZ2VzdCwgdmVyaWZ5X3BhY2thZ2UsIE5BTUVTCgoKZGVmIGlvdShhLGIpOgogICAgaW50ZXJzZWN0aW9uPW1heCgwLG1pbihhWzJdLGJbMl0pLW1heChhWzBdLGJbMF0pKSptYXgoMCxtaW4oYVszXSxiWzNdKS1tYXgoYVsxXSxiWzFdKSkKICAgIHVuaW9uPShhWzJdLWFbMF0pKihhWzNdLWFbMV0pKyhiWzJdLWJbMF0pKihiWzNdLWJbMV0pLWludGVyc2VjdGlvbgogICAgcmV0dXJuIGludGVyc2VjdGlvbi91bmlvbiBpZiB1bmlvbj4wIGVsc2UgMC4KCgpkZWYgbWF0Y2goZ3QscHJlZCx0aHJlc2hvbGQ9LjUpOgogICAgY2FuZGlkYXRlcz1zb3J0ZWQoWyhpb3UoZ1snYm94J10scFsnYm94J10pLGksaikgZm9yIGksZyBpbiBlbnVtZXJhdGUoZ3QpCiAgICAgICAgICAgICAgICAgICAgICAgZm9yIGoscCBpbiBlbnVtZXJhdGUocHJlZCkgaWYgZ1snY2xhc3MnXT09cFsnY2xhc3MnXV0scmV2ZXJzZT1UcnVlKQogICAgdXNlZF9nLHVzZWRfcD1zZXQoKSxzZXQoKQogICAgZm9yIG92ZXJsYXAsaSxqIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgaWYgb3ZlcmxhcD49dGhyZXNob2xkIGFuZCBpIG5vdCBpbiB1c2VkX2cgYW5kIGogbm90IGluIHVzZWRfcDoKICAgICAgICAgICAgdXNlZF9nLmFkZChpKTt1c2VkX3AuYWRkKGopCiAgICByZXR1cm4gdXNlZF9nLHVzZWRfcAoKCmRlZiBhbmFseXplKHJvdyxwcmVkKToKICAgIGd0PVtdCiAgICBmb3IgY2xzLHgseSx3LGggaW4gcm93WydsYWJlbHMnXToKICAgICAgICBndC5hcHBlbmQoZGljdChjbGFzc19pZD1pbnQoY2xzKSxib3g9Wyh4LXcvMikqcm93Wyd3aWR0aCddLCh5LWgvMikqcm93WydoZWlnaHQnXSwKICAgICAgICAgICAgICAgICAgICAoeCt3LzIpKnJvd1snd2lkdGgnXSwoeStoLzIpKnJvd1snaGVpZ2h0J11dKSkKICAgIGd0PVt7J2NsYXNzJzpnWydjbGFzc19pZCddLCdib3gnOmdbJ2JveCddfSBmb3IgZyBpbiBndF0KICAgIHVzZWRfZyx1c2VkX3A9bWF0Y2goZ3QscHJlZCkKICAgIGNvdW50cz17bjpkaWN0KHRwPTAsZnA9MCxmbj0wKSBmb3IgbiBpbiBOQU1FU30KICAgIG1pc3Nlcz1bXQogICAgZm9yIGksZyBpbiBlbnVtZXJhdGUoZ3QpOgogICAgICAgIGNvdW50c1tOQU1FU1tnWydjbGFzcyddXV1bJ3RwJyBpZiBpIGluIHVzZWRfZyBlbHNlICdmbiddKz0xCiAgICAgICAgaWYgaSBub3QgaW4gdXNlZF9nOgogICAgICAgICAgICBzYW1lPW1heChbaW91KGdbJ2JveCddLHBbJ2JveCddKSBmb3IgcCBpbiBwcmVkIGlmIHBbJ2NsYXNzJ109PWdbJ2NsYXNzJ11dLGRlZmF1bHQ9MCkKICAgICAgICAgICAgb3RoZXI9bWF4KFtpb3UoZ1snYm94J10scFsnYm94J10pIGZvciBwIGluIHByZWQgaWYgcFsnY2xhc3MnXSE9Z1snY2xhc3MnXV0sZGVmYXVsdD0wKQogICAgICAgICAgICBtaXNzZXMuYXBwZW5kKGRpY3QoY2xhc3NfbmFtZT1OQU1FU1tnWydjbGFzcyddXSxib3g9Z1snYm94J10sYmVzdF9zYW1lX2NsYXNzX2lvdT1zYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmVzdF9vdGhlcl9jbGFzc19pb3U9b3RoZXIsc2hvcnRfZWRnZV9hdF85NjA9bWluKGdbJ2JveCddWzJdLWdbJ2JveCddWzBdLGdbJ2JveCddWzNdLWdbJ2JveCddWzFdKSo5NjAvbWF4KHJvd1snd2lkdGgnXSxyb3dbJ2hlaWdodCddKSkpCiAgICBmb3IgaSxwIGluIGVudW1lcmF0ZShwcmVkKToKICAgICAgICBpZiBpIG5vdCBpbiB1c2VkX3A6Y291bnRzW05BTUVTW3BbJ2NsYXNzJ11dXVsnZnAnXSs9MQogICAgcmV0dXJuIGRpY3QocGFyZW50X2lkPXJvd1sncGFyZW50X2lkJ10saW1hZ2U9cm93WydpbWFnZSddLGNvdW50cz1jb3VudHMsCiAgICAgICAgICAgICAgICBtaXNzZXM9bWlzc2VzLHVubWF0Y2hlZF9wcmVkaWN0aW9ucz1bcCBmb3IgaSxwIGluIGVudW1lcmF0ZShwcmVkKSBpZiBpIG5vdCBpbiB1c2VkX3BdKQoKCmRlZiBjaGVja3BvaW50X3dlaWdodHMoYXJjaGl2ZSxvdXRwdXQpOgogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZSkgYXMgejoKICAgICAgICByZWNvcmQ9anNvbi5sb2Fkcyh6LnJlYWQoJ2NoZWNrcG9pbnRfbWFuaWZlc3QuanNvbicpKQogICAgICAgIGlmIGxlbih6Lm5hbWVsaXN0KCkpIT1sZW4oc2V0KHoubmFtZWxpc3QoKSkpIG9yIHNldCh6Lm5hbWVsaXN0KCkpIT1zZXQocmVjb3JkWydzaGEyNTYnXSl8eydjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBjaGVja3BvaW50IG1lbWJlciBsaXN0JykKICAgICAgICBmb3IgbmFtZSxleHBlY3RlZCBpbiByZWNvcmRbJ3NoYTI1NiddLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIF9faW1wb3J0X18oJ2hhc2hsaWInKS5zaGEyNTYoei5yZWFkKG5hbWUpKS5oZXhkaWdlc3QoKSE9ZXhwZWN0ZWQ6cmFpc2UgVmFsdWVFcnJvcignQ2hlY2twb2ludCBjaGVja3N1bSBtaXNtYXRjaCcpCiAgICAgICAgcGF5bG9hZD16LnJlYWQoJ3RyYWluaW5nL3dlaWdodHMvYmVzdC5wdCcpCiAgICAgICAgcGF0aD1vdXRwdXQvJ2NhbmRpZGF0ZS5wdCc7cGF0aC53cml0ZV9ieXRlcyhwYXlsb2FkKQogICAgICAgIHJldHVybiBwYXRoLGpzb24ubG9hZHMoei5yZWFkKCdzaWduYXR1cmUuanNvbicpKQoKCmRlZiBydW4ocGFja2FnZSxjaGVja3BvaW50LG91dHB1dCxkZXZpY2U9J2NwdScpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBwYWNrYWdlLG91dHB1dD1QYXRoKHBhY2thZ2UpLnJlc29sdmUoKSxQYXRoKG91dHB1dCkucmVzb2x2ZSgpCiAgICBpZiBvdXRwdXQuZXhpc3RzKCk6cmFpc2UgVmFsdWVFcnJvcignVXNlIGEgbmV3IGRpYWdub3N0aWNzIG91dHB1dCBmb2xkZXI7IGV4aXN0aW5nIHJlc3VsdHMgcHJlc2VydmVkLicpCiAgICB2ZXJpZnlfcGFja2FnZShwYWNrYWdlKTtvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlKQogICAgY2FuZGlkYXRlLHNpZ25hdHVyZT1jaGVja3BvaW50X3dlaWdodHMoY2hlY2twb2ludCxvdXRwdXQpCiAgICBpZiBzaWduYXR1cmUuZ2V0KCdpZGVudGl0eScse30pLmdldCgncGFja2FnZV9zaGEyNTYnKSE9ZGlnZXN0KHBhY2thZ2UvJ3BhY2thZ2VfaGFzaGVzLmpzb24nKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50L3BhY2thZ2UgbWlzbWF0Y2gnKQogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDQpCiAgICBtYW5pZmVzdD1qc29uLmxvYWRzKChwYWNrYWdlLydtYW5pZmVzdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICByb3dzPVtyIGZvciByIGluIG1hbmlmZXN0Wyd2aWV3cyddIGlmIHJbJ2RhdGFzZXQnXT09J3NvdXJjZScgYW5kIHJbJ3NwbGl0J109PSd2YWwnIGFuZCByWydraW5kJ109PSdmdWxsJ10KICAgIHN1bW1hcnk9eydzY29wZSc6J1NvdXJjZSB2YWxpZGF0aW9uIG9ubHkuIEZpeGVkIGNvbmY9MC4yNSwgSW9VPTAuNSwgZ3JlZWR5IG9uZS10by1vbmUgc2FtZS1jbGFzcyBtYXRjaGluZy4gVGhlc2UgY291bnRzIGFyZSBub3QgbUFQIG9yIHJlcG9ydGVkIG9wdGltYWwtdGhyZXNob2xkIHByZWNpc2lvbi9yZWNhbGwuJywKICAgICAgICAgICAgICdpbWFnZXMnOmxlbihyb3dzKSwnY2hlY2twb2ludF9zaGEyNTYnOmRpZ2VzdChjaGVja3BvaW50KSwnbW9kZWxzJzp7fX0KICAgIGZvciBuYW1lLHdlaWdodHMgaW4gWygnYmFzZWxpbmUnLHBhY2thZ2UvJ3dlaWdodHMvc291cmNlX2Jhc2VsaW5lLnB0JyksKCdjYW5kaWRhdGUnLGNhbmRpZGF0ZSldOgogICAgICAgIG1vZGVsPVlPTE8oc3RyKHdlaWdodHMpKQogICAgICAgIGlmIGxpc3QobW9kZWwubmFtZXMudmFsdWVzKCkpIT1OQU1FUzpyYWlzZSBWYWx1ZUVycm9yKCdDbGFzcyBvcmRlciBkaWZmZXJzJykKICAgICAgICBkZXRhaWxzPVtdO2NvdW50cz17bjpkaWN0KHRwPTAsZnA9MCxmbj0wKSBmb3IgbiBpbiBOQU1FU30KICAgICAgICBmb3IgaW5kZXgscm93IGluIGVudW1lcmF0ZShyb3dzLDEpOgogICAgICAgICAgICByZXN1bHQ9bW9kZWwucHJlZGljdChzdHIocGFja2FnZS9yb3dbJ2ltYWdlJ10pLGltZ3N6PTk2MCxjb25mPS4yNSxpb3U9LjcsbWF4X2RldD0zMDAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZT1kZXZpY2UsaGFsZj1GYWxzZSxyZWN0PUZhbHNlLHZlcmJvc2U9RmFsc2UpWzBdCiAgICAgICAgICAgIHByZWQ9W3snY2xhc3MnOmludChjKSwnYm94Jzpib3gsJ2NvbmZpZGVuY2UnOmZsb2F0KGNvbmYpfSBmb3IgYyxib3gsY29uZiBpbgogICAgICAgICAgICAgICAgICB6aXAocmVzdWx0LmJveGVzLmNscy5jcHUoKS50b2xpc3QoKSxyZXN1bHQuYm94ZXMueHl4eS5jcHUoKS50b2xpc3QoKSxyZXN1bHQuYm94ZXMuY29uZi5jcHUoKS50b2xpc3QoKSldCiAgICAgICAgICAgIGRldGFpbD1hbmFseXplKHJvdyxwcmVkKTtkZXRhaWxzLmFwcGVuZChkZXRhaWwpCiAgICAgICAgICAgIGZvciBuIGluIE5BTUVTOgogICAgICAgICAgICAgICAgZm9yIGsgaW4gWyd0cCcsJ2ZwJywnZm4nXTpjb3VudHNbbl1ba10rPWRldGFpbFsnY291bnRzJ11bbl1ba10KICAgICAgICAgICAgaWYgaW5kZXglMjA9PTA6cHJpbnQobmFtZSxpbmRleCwnLycsbGVuKHJvd3MpLGZsdXNoPVRydWUpCiAgICAgICAgKG91dHB1dC9mJ3tuYW1lfV9lcnJvcnMuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkZXRhaWxzLGluZGVudD0yKSkKICAgICAgICBzdW1tYXJ5Wydtb2RlbHMnXVtuYW1lXT17J3dlaWdodHNfc2hhMjU2JzpkaWdlc3Qod2VpZ2h0cyksJ2NvdW50cyc6Y291bnRzfQogICAgICAgIGRlbCBtb2RlbAogICAgKG91dHB1dC8nZGlhZ25vc2lzLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSxpbmRlbnQ9MikrJ1xuJykKICAgIGFyY2hpdmU9b3V0cHV0LydkaWFnbm9zaXMuemlwJwogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZSwndycsemlwZmlsZS5aSVBfREVGTEFURUQpIGFzIHo6CiAgICAgICAgZm9yIG5hbWUgaW4gWydkaWFnbm9zaXMuanNvbicsJ2Jhc2VsaW5lX2Vycm9ycy5qc29uJywnY2FuZGlkYXRlX2Vycm9ycy5qc29uJ106ei53cml0ZShvdXRwdXQvbmFtZSxuYW1lKQogICAgcHJpbnQoanNvbi5kdW1wcyhzdW1tYXJ5LGluZGVudD0yKSxmbHVzaD1UcnVlKQogICAgcmV0dXJuIHN1bW1hcnkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgcGFyc2VyPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXBhY2thZ2UnLHR5cGU9UGF0aCxyZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1jaGVja3BvaW50Jyx0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tb3V0cHV0Jyx0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tZGV2aWNlJyxkZWZhdWx0PSdjcHUnKQogICAgYXJncz1wYXJzZXIucGFyc2VfYXJncygpCiAgICBvcy5lbnZpcm9uLnNldGRlZmF1bHQoJ1lPTE9fQ09ORklHX0RJUicsc3RyKGFyZ3Mub3V0cHV0LnBhcmVudC8nZGlhZ25vc2lzX3lvbG8nKSkKICAgIHJ1bihhcmdzLnBhY2thZ2UsYXJncy5jaGVja3BvaW50LGFyZ3Mub3V0cHV0LGFyZ3MuZGV2aWNlKQo=', 'sha256': 'd6af6d4db4db4b0e1c3e983b5be2f9ca3d94c7c7ac3febedfedde990307c5061'}}

import base64, importlib
TOOLS = Path('/kaggle/working/pcb_diagnosis_tools')
(TOOLS/'revamp').mkdir(parents=True, exist_ok=True)
for name, record in EMBEDDED_TOOLS.items():
    raw = base64.b64decode(record['base64'])
    assert hashlib.sha256(raw).hexdigest() == record['sha256']
    destination = TOOLS/'revamp'/name
    if destination.exists() and destination.read_bytes() != raw:
        raise RuntimeError('Different resume code already installed; use a fresh kernel/session.')
    if not destination.exists(): destination.write_bytes(raw)
sys.path.insert(0, str(TOOLS))
# A previous notebook import can remain cached after rerunning a cell.
for name in list(sys.modules):
    if name == 'revamp' or name.startswith('revamp.'): del sys.modules[name]
importlib.invalidate_caches()
from revamp.runtime import verify_package, configure
print(verify_package(WORK))
CONFIGS = configure(WORK, Path('/kaggle/working/pcb_phase2_configs'))
print('Setup verified; existing matching files reused.', CONFIGS)


In [ ]:
import platform, ultralytics
# Import failures here should be resolved before training; never silently change torch.
from ultralytics import YOLO
environment = {'python':platform.python_version(), 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'ultralytics':ultralytics.__version__, 'cuda_available':torch.cuda.is_available(), 'cuda':torch.version.cuda}
Path('/kaggle/working/pcb_environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
for name in ['yolov8n', 'yolov8s']:
    assert len(YOLO(str(WORK/f'weights/{name}.pt')).names) == 80
for name in ['source', 'target']:
    assert list(YOLO(str(WORK/f'weights/{name}_baseline.pt')).names.values()) == json.loads((WORK/'manifest.json').read_text())['names']

assert torch.cuda.is_available(), 'Enable GPU T4 before running source training.'
environment['gpu'] = torch.cuda.get_device_name(0)
Path('/kaggle/working/pcb_environment.json').write_text(json.dumps(environment, indent=2))


In [ ]:
CHECKPOINT = Path('/kaggle/working/pcb_phase3_source_full_boards_v2/full_boards_run_checkpoint.zip')
if not CHECKPOINT.is_file():
    matches = list(Path('/kaggle/input').rglob('full_boards_run_checkpoint.zip'))
    if len(matches) != 1: raise RuntimeError('Set CHECKPOINT to the trusted downloaded full-board checkpoint ZIP.')
    CHECKPOINT = matches[0]
RUN = Path('/kaggle/working/pcb_source_diagnosis_v2')
if RUN.exists(): raise RuntimeError('Output exists; download its diagnosis.zip or choose a new RUN folder.')
import signal
from IPython.display import display, FileLink
command = [sys.executable, '-m', 'revamp.diagnose', '--package', str(WORK), '--checkpoint', str(CHECKPOINT), '--output', str(RUN), '--device', '0']
env = dict(os.environ, PYTHONPATH=str(TOOLS), MPLCONFIGDIR='/kaggle/working/pcb_mpl')
process = subprocess.Popen(command, cwd=TOOLS, env=env, start_new_session=True)
try:
    if process.wait(): raise RuntimeError('Diagnostic failed; see output above. No training was performed.')
except KeyboardInterrupt:
    try: os.killpg(process.pid, signal.SIGINT)
    except ProcessLookupError: pass
    try: process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
finally:
    if (RUN/'diagnosis.zip').exists(): display(FileLink(os.path.relpath(RUN/'diagnosis.zip', Path.cwd())))


Download **diagnosis.zip** and keep the four original training outputs. Save them in `downloads/source_full_boards_v2/`. The ZIP contains per-image and per-class error counts for both models. No model is promoted or retrained.